# E23 — Dual-path cross-attention (strong / weak frequency paths) + wavelet-scattering audio tokens

**Context.** The setup: TII UAV Realistic Fault Dataset; 5-class propeller-fault severity; 95 missions; 5 days;
leave-one-day-out.

**What is known:**
- Day-out window F1: IMU-only ≈ 0.84, audio-only ≈ 0.80.
- E18b confirmed that IMU-anchored residual fusion (R1) beats its controls (C1: IMU features only; C2: shuffled
  audio) by ≈ +0.02, on 5/5 days and 5/5 seeds. About two thirds of R1's gain over the IMU model comes from the extra
  head, not from audio.
- E16, E19 and E22 were all NOT CREDITED.

**E23 tests the guide's proposal:**
- **Dual-path cross-attention.** Strong-energy frequency components of audio and IMU go to one multi-head attention
  path, and weak ones to the other.
- **Wavelet scattering** features as an alternative audio token.

It is a last, honest test of whether a more structured audio branch raises the **audio-specific** contribution above
R1. No deployment assumptions: audio is always the plain N0 version.

| arm | what it is |
|---|---|
| B0 | IMU-ONLY anchor (E18b code; frozen in every residual arm) |
| C1 | IMU-only correction head (E18b code; reference) |
| **R1** | single-path residual fusion, AudioMamba on R0 tokens (E18b code) — **the reference for the primary contrasts** |
| **D1** | DUAL-PATH, FFT (R0) tokens, energy-based strong/weak split |
| D1r | DUAL-PATH, FFT tokens, RANDOM split of the same sizes (control: "two paths" vs "strong/weak criterion") |
| **D2** | DUAL-PATH, WAVELET-SCATTERING tokens, energy-based split |
| D1sh / D2sh | D1 / D2 with audio shuffled within each split (E18b's C2 rule) — the audio-attribution controls |

**Windows and seeds:**
- The **full** v29 paired window set (5,238 windows, as in E18b; no reference flights removed).
- E18b's folds and inner validation (fallback per-class hold-out, seed 2026).
- Seeds 11, 21, 31, 41, 51.
- Numbers are comparable with E18b, not with E22.

In [ ]:
# ==============================================================================
# E23 RUN FLAGS (the only switches a user needs)
# ==============================================================================
SMOKE = False          # True: fold 1, seed 11, 2 epochs, every arm -> *_smoke outputs, no analysis
RUN_SUBSET = None      # None = everything; or e.g. {"arms": ["B0", "D1"], "folds": [0, 1], "seeds": [11]}
                       # (missing dependencies are added automatically and printed: every residual / dual-path arm needs B0)
RESTORE_FROM = None    # optional: attached earlier E23 output(s): one folder or a list, e.g. "/kaggle/input/<name>".
                       # Its e23_results.csv + the listed e23_probs/*.npz and e23_checkpoints/*.pt are copied into
                       # /kaggle/working, so finished runs are skipped. Search depth <= 3 (never walks dataset clones).
# ---- SEED CHUNKING (one "Save Version" per chunk; each chunk attaches the PREVIOUS chunk's output) ----
#   chunk 1:  RUN_SUBSET = {"seeds": [11]}   RESTORE_FROM = None
#   chunk 2:  RUN_SUBSET = {"seeds": [21]}   RESTORE_FROM = "/kaggle/input/<output of chunk 1>"
#   chunk 3:  RUN_SUBSET = {"seeds": [31]}   RESTORE_FROM = "/kaggle/input/<output of chunk 2>"
#   chunk 4:  RUN_SUBSET = {"seeds": [41]}   RESTORE_FROM = "/kaggle/input/<output of chunk 3>"
#   chunk 5:  RUN_SUBSET = {"seeds": [51]}   RESTORE_FROM = "/kaggle/input/<output of chunk 4>"
#   final:    RUN_SUBSET = {"seeds": []}     RESTORE_FROM = "/kaggle/input/<output of chunk 5>"   <- analysis only
#   Each chunk's output contains everything restored + its own runs, so attaching only the latest one is enough.
#   A chunk ends normally ("analysis needs all seeds") until the final, analysis-only chunk.
CLEAN_DISK_AFTER_TRAINING = True   # after ALL planned runs are done: delete the parse cache and the raw dataset clone
                                   # from /kaggle/working (several GB; not needed by the analysis)
E23_BUILD = "E23 build v1 (2026-10-15): dual-path cross-attention + wavelet scattering, full v29 windows, E23 seeds"
print("=" * 110)
print(E23_BUILD)
print("If this line is missing or shows an older build, Kaggle is running an OLD copy of the notebook: re-upload it.")
print("=" * 110, flush=True)

## 0. Setup (E18b, verbatim except output names e18b_ → e23_)

In [ ]:
# ==============================================================================
# 0a. ENVIRONMENT, PATHS, DATASET
# ==============================================================================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must be set before the first cuBLAS call (determinism)

import ast
import copy
import csv
import gc
import glob
import hashlib
import json
import math
import random
import re
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
if IS_KAGGLE:
    BASE, OUT = Path("/kaggle/working/tii"), Path("/kaggle/working")
elif IS_COLAB:
    BASE, OUT = Path("/content/tii"), Path("/content")
else:
    BASE, OUT = Path.cwd() / "tii", Path.cwd() / "e23_outputs"
BASE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
ENV = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local")

REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p: Path) -> bool:
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_attached = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _attached += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _attached:
    DATA_ROOT, DATA_SOURCE = _attached[0], "attached Kaggle dataset"
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) into {_repo} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT, DATA_SOURCE = _repo / "Dataset", "git clone --depth 1"
assert _is_dataset_root(DATA_ROOT), f"{DATA_ROOT} does not contain class folders 0..4"

FIG_DIR = OUT / "figures"
CACHE_DIR = OUT / "e23_audio_cache"          # E18: own cache dir
PROBS_DIR = OUT / "e23_probs"
CKPT_DIR = OUT / "e23_checkpoints"
RESULTS_CSV = OUT / "e23_results.csv"
for _d in (FIG_DIR, CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV} | BASE {BASE} | outputs {OUT}")
print(f"Dataset: {DATA_ROOT} ({DATA_SOURCE})")
print(f"Figures -> {FIG_DIR} | cache -> {CACHE_DIR} | results -> {RESULTS_CSV} | probs -> {PROBS_DIR}")

In [ ]:
# ==============================================================================
# 0b. REAL mamba-ssm (no fallback). Installs only if it is not importable already.
# ==============================================================================
import importlib
import importlib.util

import torch


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Real mamba-ssm needs a GPU: enable a GPU accelerator (Kaggle: Settings -> "
                       "Accelerator -> GPU) and re-run. There is deliberately no CPU fallback in this notebook.")

if importlib.util.find_spec("mamba_ssm") is None:
    print(f"Installing mamba-ssm (torch {torch.__version__}, CUDA {torch.version.cuda}, python "
          f"{sys.version_info.major}.{sys.version_info.minor}) ...")
    _ok = (_pip(["packaging", "ninja"])
           and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output above). Check that internet is ON and a GPU "
                           "is attached, then restart the session and re-run. No fallback model is used.")

try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"mamba_ssm could not be imported ({exc!r}). Restart the kernel after installation and re-run; "
                       "no fallback model is used.") from exc
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} imported: {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | "
      f"torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 0c. CONSTANTS — every fixed value lives here
# ==============================================================================
SEEDS = [42, 123, 2024]

# --- AudioBuffer layout: 1536 = 256 frames x 6 interleaved slots; sample i -> frame i // 6, slot i % 6 ---
N_SLOTS = 6
DROP_SLOT = 5                                   # 0-based slot index 5 (the 6th slot): identically zero, dropped
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
N_CH = len(KEEP_SLOTS)                          # 5 audio channels
FRAME = 256                                     # frames per buffer = samples per channel per buffer
BUFFER_LEN = N_SLOTS * FRAME                    # 1536
N_RFFT = FRAME // 2 + 1                         # 129 per-channel rFFT bins (0..128)
KEEP_BINS = list(range(1, 17))                  # model input: per-channel bins 1..16 (fixed a priori)
N_FEAT = N_CH * len(KEEP_BINS)                  # 80

# --- windows (chunk counts only: no IMU, no time axis) ---
WINDOW_CHUNKS = 128
STRIDE_CHUNKS = 64
TA = 64                                         # tokens per window
TOKEN_POOL = WINDOW_CHUNKS // TA                # 2 consecutive chunks averaged per token
STEADY = (0.10, 0.90)                           # steady flight = 10%..90% of a mission's chunks (by position)

ASSUMED_CHANNEL_RATE_HZ = 16000                 # UNVERIFIED, used ONLY for secondary axis labels, never in any computation

# --- dataset facts (asserted in section 1) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5                             # exclude if audio_start_offset / mission_duration >= this
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}        # the single remaining 12-29 mission joins the 12-31 group
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"

# --- model + training: transcribed from version 28/minor-project-v28.ipynb (config cell, UnimodalMamba, train_unimodal),
#     the audio arms A1/A2. E19: used unchanged; the v28 notebook is not read. ---
D_MODEL = 256
DROPOUT = 0.15
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496                 # 2 x 437,248 (real mamba_ssm, incl. the pre-norm LayerNorms)
EPOCHS = 20                                     # fixed; no early stopping in v28
BATCH_SIZE = 32
LEARNING_RATE = 1e-4                            # AdamW
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6                                   # CosineAnnealingLR(T_max=EPOCHS, eta_min=LR_MIN), stepped per epoch
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
# checkpoint rule (v28): after every epoch, keep the state with the highest validation macro-F1 (strictly greater)

# --- splits ---
INNER_VAL_FRACTION, INNER_VAL_SEED = 0.15, 2027                    # v27 _split_inner_val (day-out protocol index 0)
INNER_VAL_FALLBACK_SEED = 2026                                     # used ONLY if v27's notebook cannot be read
V27_INNER_VAL_SHA1 = "58e3c037c513ace2"         # sha1[:16] of v27's _split_inner_val source (ast segment)

# --- analysis (fixed before seeing results) ---
N_PERMUTATIONS = 1000
PERM_SEED = 2029
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
TOP_CELLS = 10
SIGN_CONSISTENT_MIN_DAYS = 4
MIN_PER_CLASS_FOR_DAY_SIGN = 2
SPEARMAN_THRESHOLD = 0.7
PCA_BINS = list(range(1, 33))
EVIDENCE_K = [1, 2, 4, 8, 16, 32]

CHECK_DETERMINISM = False                       # E19: no hard determinism gate (mamba-ssm CUDA is non-deterministic); A0 vs reference is a soft check
DETERMINISM_EPOCHS = 3

HZ_PER_BIN = ASSUMED_CHANNEL_RATE_HZ / FRAME    # 62.5 Hz/bin — secondary axis labels ONLY
print(f"{N_SLOTS} slots x {FRAME} frames = {BUFFER_LEN} samples per buffer | channels kept {[s + 1 for s in KEEP_SLOTS]} "
      f"(slot {DROP_SLOT + 1} dropped) | model input {N_CH} ch x bins {KEEP_BINS[0]}-{KEEP_BINS[-1]} = {N_FEAT} features")
print(f"Windows: {WINDOW_CHUNKS} chunks, stride {STRIDE_CHUNKS}, {TA} tokens of {TOKEN_POOL} chunks | seeds {SEEDS}")

In [ ]:
# ==============================================================================
# 0c-IMU. IMU reference constants (Ran_TII_IMU_Only_Analysis_Mamba.ipynb, cell 'CONSTANTS', verbatim lines) +
#         a check that every setting SHARED with the audio reference is identical
# ==============================================================================
CHANNELS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
WIN_S = 2.56                     # WINDOW_SEC
STRIDE_S = 1.28                  # STEP_SEC
TS = 128                         # TS_SENSOR: samples per window, chosen evenly BY INDEX among the window's file-order samples
MIN_SAMPLES_PER_WINDOW = 10      # MIN_SENSOR_SAMPLES_PER_WINDOW
SENSOR_STD_EPS = 1e-6

IMU_REFERENCE_SHARED = {'SEEDS': [42, 123, 2024], 'N_CLASSES': 5, 'EXPECTED_EXCLUDED': {'rosbag2_2021_12_31-10_04_22', 'rosbag2_2021_12_31-09_58_13', 'rosbag2_2021_12_29-09_10_31', 'rosbag2_2021_12_31-10_12_31'}, 'EXPECTED_N_MISSIONS': 95, 'EXPECTED_CLASS_COUNTS': [18, 19, 18, 20, 20], 'DAYS': ['2021-12-22', '2021-12-31', '2022-01-04', '2022-01-05', '2022-01-10'], 'DAY_MERGE': {'2021-12-29': '2021-12-31'}, 'EXPECTED_DAY_COUNTS': [10, 36, 17, 15, 17], 'SMALL_FOLD_DAY': '2021-12-22', 'STEADY': (0.1, 0.9), 'D_MODEL': 256, 'DROPOUT': 0.15, 'MAMBA_NUM_BLOCKS': 2, 'MAMBA_D_STATE': 16, 'MAMBA_D_CONV': 2, 'MAMBA_EXPAND': 2, 'EXPECTED_MAMBA_PARAMS': 874496, 'EPOCHS': 20, 'BATCH_SIZE': 32, 'LEARNING_RATE': 0.0001, 'WEIGHT_DECAY': 0.0001, 'LR_MIN': 1e-06, 'GRAD_CLIP_NORM': 1.0, 'NUM_WORKERS': 2, 'INNER_VAL_FRACTION': 0.15, 'INNER_VAL_SEED': 2027, 'INNER_VAL_FALLBACK_SEED': 2026, 'V27_INNER_VAL_SHA1': '58e3c037c513ace2', 'BOOTSTRAP_RESAMPLES': 2000, 'BOOTSTRAP_SEED': 2028}
_diff = {k: (globals()[k], v) for k, v in IMU_REFERENCE_SHARED.items() if globals()[k] != v}
assert not _diff, f"IMU and audio reference settings differ: {_diff}"
print(f"IMU constants: windows {WIN_S} s / stride {STRIDE_S} s, Ts {TS}, channels {CHANNELS}; all "
      f"{len(IMU_REFERENCE_SHARED)} settings shared with the audio reference are identical")


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def find_file(fname, subdirs):
    cands = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        cands += [str(base / s / fname) for s in subdirs] + [str(base / fname)]
    return next((Path(c) for c in cands if Path(c).is_file()), None)

In [ ]:
# ==============================================================================
# 0c-v29. PAIRED-WINDOW CONSTANTS (version 29/minor-project-v29.ipynb, cell 2c, verbatim lines) + equality checks
# ==============================================================================
N_AUDIO_FEAT = len(KEEP_SLOTS) * len(KEEP_BINS)  # 80
AUDIO_STD_EPS = 1e-6
WINDOW_SEC = 2.56
STEP_SEC = 1.28
TA_AUDIO = 64
TS_SENSOR = 128
MIN_SENSOR_SAMPLES_PER_WINDOW = 10
N_SENSOR_CHANNELS = 6
N_HASH_SAMPLE = 25

assert (WINDOW_SEC, STEP_SEC, TS_SENSOR, MIN_SENSOR_SAMPLES_PER_WINDOW, N_SENSOR_CHANNELS) == \
       (WIN_S, STRIDE_S, TS, MIN_SAMPLES_PER_WINDOW, len(CHANNELS)), "v29 IMU window constants differ from the IMU reference"
assert (N_AUDIO_FEAT, TA_AUDIO) == (N_FEAT, TA), "v29 audio constants differ from the audio reference (R0: 80 features, 64 tokens)"
print(f"Paired windows: {WINDOW_SEC} s, step {STEP_SEC} s | IMU {TS_SENSOR} x {N_SENSOR_CHANNELS} | audio {TA_AUDIO} x "
      f"{N_AUDIO_FEAT} (R0 features) — identical to the IMU and audio references")

In [ ]:
# ==============================================================================
# 0c-E23. E23 CONSTANTS (E18b's, with the E23 arms, rule and budget) — ALL fixed a priori
# ==============================================================================
E18_SEEDS = [42, 123, 2024, 7, 99]                 # the seeds E18 used
SEEDS = [11, 21, 31, 41, 51]                       # E23: the SAME seeds as E18b
assert not set(SEEDS) & set(E18_SEEDS), "an E18b seed equals an E18 seed"
print("NOTE: E23 uses the full v29 window set and the E23 seeds: its numbers are comparable with E18b (same 95 "
      "missions and 5 days, so the seeds test training randomness only, not new data).")
ARMS = ["B0", "C1", "R1", "D1", "D1r", "D2", "D1sh", "D2sh"]   # run order inside every (fold, seed)
CANDIDATES = ["D1", "D2"]                          # E23: primary contrasts vs R1 (K = 2)
CONTROLS = ["C1", "D1r", "D1sh", "D2sh"]
ARM_NAMES = {"B0": "IMU-ONLY (anchor)", "C1": "IMU-only correction head (reference)",
             "R1": "single-path residual, AudioMamba on R0 (reference arm)", "D1": "DUAL-PATH, FFT tokens, energy split",
             "D1r": "DUAL-PATH, FFT tokens, RANDOM split", "D2": "DUAL-PATH, scattering tokens, energy split",
             "D1sh": "D1 with shuffled audio", "D2sh": "D2 with shuffled audio"}
DEPENDS = {a: ["B0"] for a in ["C1", "R1", "D1", "D1r", "D2", "D1sh", "D2sh"]}
CKPT_ARMS = {"B0"}                                  # E23: only the anchor is loaded by later arms

# ---- residual fusion (fixed; never tuned) ----
ALPHA_MAX = 1.0                                    # alpha = ALPHA_MAX * sigmoid(g(.)) in [0, ALPHA_MAX]
LAMBDA_R = 0.01                                    # weight of mean ||alpha * dz||^2
LAMBDA_I = 0.5                                     # R2 only: weight of CE(z_IMU)
R_HIDDEN = 128                                     # W1: Linear(d_in, 128) -> ReLU -> W2: Linear(128, 5) (zero-init)
R2_EPOCHS = 10
R2_IMU_LR_FACTOR = 0.1
B1_WEIGHTS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]        # ties on validation F1 -> the larger w (closer to IMU-only)
C2_PERM_SEED = 2031
ZERO_INIT_TOL = 1e-5
ANCHOR_REPRO_TOL = 1e-3                            # max |p_anchor - p_B0| on test windows (same weights, eval mode)
PAIRING_MIN_COVERAGE = 0.90

# ---- pre-registered rule ----
MIN_DELTA = 0.02
PAIRED_CI = 99.0
MIN_FOLDS_SAME_SIGN = 4
MIN_SEEDS_POSITIVE = 4
NO_HARM_C4_RECALL = 0.05
NO_HARM_FCN = 0.02
RULE_TEXT = 'PRE-REGISTERED DECISION RULE (E23; fixed before any E23 result; applied in code; never edited after results)\nPrimary metric: pooled out-of-fold WINDOW macro-F1 from seed-averaged probabilities on identical windows (the full v29\npaired window set, 95 missions, day-out, seeds 11/21/31/41/51 — comparable with E18b).\nPRIMARY CONTRASTS (K = 2): D1 vs R1 and D2 vs R1. A candidate is CREDITED only if ALL hold:\n  1. pooled paired delta vs R1 >= +0.01 (the audio-specific part of R1 is only ~+0.02; each contrast is also reported\n     against the stricter +0.02 line, descriptively);\n  2. the 99% paired mission-cluster bootstrap CI (2000 resamples of missions within class, seed 2028, the SAME\n     resampled missions for both arms) excludes 0;\n  3. delta > 0 in >= 4 of 5 folds (seed-averaged, per held-out day);\n  4. delta > 0 in >= 4 of 5 seeds (each seed\'s own pooled out-of-fold F1);\n  5. worst-day F1 not below R1\'s;\n  6. NO-HARM: Class-4 recall not more than 0.05 below R1\'s, and "faulty called Normal" not more than 0.02 above R1\'s;\n  7. AUDIO ATTRIBUTION: the candidate beats its OWN shuffled control (D1 vs D1sh, D2 vs D2sh) by >= +0.02 with the\n     99% CI excluding 0, >= 4/5 folds and >= 4/5 seeds.\nWORSE (mirror): delta <= -0.01, the 99% CI entirely below 0, and delta < 0 in >= 4 of 5 folds. Otherwise NOT CREDITED.\nSECONDARY (reported, conditions 1-5, no multiplicity claim): D2 vs D1 (scattering vs FFT); D1 vs D1r (energy split vs\nrandom split); R1 vs C1, D1 vs C1, D2 vs C1, D1sh vs C1, D2sh vs C1 (capacity of the new head by itself; does R1\nstill replicate +0.02 over C1).\nAlso reported: per-day F1 for all arms, per-class recall, residual diagnostics (corrected / broken / net, mean alpha),\nand the eval-time PATH ABLATION for D1 and D2 (zero the strong-path output, then the weak-path output; F1 change) as a\ndiagnostic of which path carries the signal. Every arm is reported whatever the outcome; a null result is valid.\nNo deployment assumptions: audio is always the plain N0 version.'

# ---- soft check vs E19 A0 (IMU) and the runtime budget ----
E19_A0_FALLBACK = [0.8324, 0.8489, 0.7260, 0.8316, 0.8129]   # IMU reference per-fold F1 if no e19_results.csv is found
E18_B0_PER_FOLD = [0.9085, 0.8194, 0.6465, 0.8944, 0.9018]   # E18 table: B0 per-fold means
B0_REF_TOL = 0.05
SEC_PER_RUN = {"B0": 62, "C1": 35, "R1": 80, "D1": 95, "D1r": 95, "D2": 100, "D1sh": 95, "D2sh": 100}
# (B0/C1/R1 from E18/E18b on a Kaggle T4; the D arms are guesses: one attention layer per path + a frozen anchor)
BUDGET_HOURS = 3.5

# ---- run plan ----
if SMOKE:
    FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN, R2_EPOCHS_TO_RUN = [1], [11], 2, 2     # E18b: smoke uses seed 11
    RESULTS_CSV = OUT / "e23_results_smoke.csv"
    PROBS_DIR = OUT / "e23_probs_smoke"
    CKPT_DIR = OUT / "e23_checkpoints_smoke"
else:
    FOLDS_TO_RUN, SEEDS_TO_RUN, EPOCHS_TO_RUN, R2_EPOCHS_TO_RUN = list(range(len(DAYS))), SEEDS, EPOCHS, R2_EPOCHS
_sub = RUN_SUBSET or {}
FOLDS_TO_RUN = _sub.get("folds", FOLDS_TO_RUN)
SEEDS_TO_RUN = _sub.get("seeds", SEEDS_TO_RUN)
assert set(SEEDS_TO_RUN) <= set(SEEDS), f"RUN_SUBSET seeds {SEEDS_TO_RUN} must be E23 seeds {SEEDS}"
ARMS_TO_RUN = list(_sub.get("arms", ARMS))
_added = True
while _added:
    _added = False
    for a in list(ARMS_TO_RUN):
        for dep in DEPENDS.get(a, []):
            if dep not in ARMS_TO_RUN:
                ARMS_TO_RUN.append(dep)
                _added = True
                print(f"RUN_SUBSET: added dependency {dep} (needed by {a})")
ARMS_TO_RUN = [a for a in ARMS if a in ARMS_TO_RUN]
assert set(ARMS_TO_RUN) <= set(ARMS)
for _d in (PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
(OUT / ("e23_rule_smoke.txt" if SMOKE else "e23_rule.txt")).write_text(RULE_TEXT + "\n", encoding="utf-8")
print(f"SMOKE={SMOKE} | arms {ARMS_TO_RUN} | folds {FOLDS_TO_RUN} | seeds {SEEDS_TO_RUN} | epochs {EPOCHS_TO_RUN} "
      f"(R2 +{R2_EPOCHS_TO_RUN}) | results -> {RESULTS_CSV.name} | probs -> {PROBS_DIR.name}/ | rule -> e23_rule.txt")

# ---- E23: dual-path module, scattering and rule constants (fixed a priori, never tuned on any fold) ----
DUAL_ARMS = ["D1", "D1r", "D2", "D1sh", "D2sh"]
PATHS = ("strong", "weak")
DUAL_D_MODEL = 64                                 # tokenisers -> d_model 64
DUAL_HEADS = 4                                    # multi-head cross-attention, 4 heads
DUAL_POS_INIT_STD = 0.02                          # learned positional embedding init
D1R_SEED = 2033                                   # D1r: random strong/weak assignment, seed [D1R_SEED, fold]
SCAT_J, SCAT_Q, SCAT_MAX_ORDER = 6, 2, 2          # Scattering1D(J=6, Q=2, max_order=2, T=2^J)
SCAT_EPS = 1e-3                                   # log(1 + S / 1e-3)
SCAT_BATCH_KYMATIO, SCAT_BATCH_OWN = 4096, 512    # segments per GPU batch
MORLET_XI_MAX = 0.35                              # fallback Morlet bank: highest centre frequency (cycles/sample)
MIN_DELTA_PRIMARY = 0.01                          # condition 1 (and the WORSE mirror)
MIN_DELTA_STRICT = 0.02                           # descriptive line
ATTR_MIN_DELTA = 0.02                             # condition 7: candidate vs its own shuffled control
E18B_REF_TOL = 0.06                               # soft hook 12

In [ ]:
# ==============================================================================
# 0d. SEEDING / DETERMINISM, FILE-ACCESS LOGGER, PASS/FAIL HOOKS, FIGURE HELPERS
# ==============================================================================
def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed: int) -> torch.Generator:
    """DataLoader shuffling uses its own generator, so the shuffle order depends only on the seed."""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id: int) -> None:
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
set_all_seeds(SEEDS[0])
print("Determinism settings: cudnn.deterministic=True, cudnn.benchmark=False, "
      "torch.use_deterministic_algorithms(True, warn_only=True) (warn_only because mamba-ssm's custom CUDA kernels are "
      f"outside PyTorch's deterministic registry), CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']}, "
      "per-run seeding of random/numpy/torch/CUDA, a seeded torch.Generator for DataLoader shuffling, and worker_init_fn.")

# ---- dataset-file access logger: EVERY dataset file read in this notebook goes through open_logged ----
ACCESS_KINDS = ("AudioBuffer:first_record", "AudioBuffer:full_parse", "SensorCombined:first_last_timestamp")
FILE_LOG = Counter()
FILE_LOG_PER_MISSION = {k: Counter() for k in ACCESS_KINDS}


def open_logged(path, kind, mode="r"):
    assert kind in ACCESS_KINDS, f"unknown access kind {kind}"
    FILE_LOG[kind] += 1
    FILE_LOG_PER_MISSION[kind][Path(path).parent.name] += 1
    return open(path, mode) if "b" in mode else open(path, mode, encoding="utf-8")


# ---- verification hooks ----
HOOKS = defaultdict(list)
HOOK_NAMES = {1: "data: exclusion set, mission/class/day counts, slot 6 zero (v29 metadata)",
              2: "pairing: audio + IMU from the same span of one mission; hash rebuild; shapes (v29)",
              3: "split leakage: test day absent from fit/val; inner val has all 5 classes",
              4: "pairing coverage: >= 90% of paired windows hold real audio chunks (IMU-grid coverage printed)",
              5: "identical test window ids / missions / training pool across all arms per (fold, seed)",
              6: "real mamba_ssm + params", 7: "residual zero-init: z_final == z_IMU on the first batch before any update",
              8: "frozen anchor: IMU parameters unchanged by every residual arm; anchor reproduces B0's test probabilities",
              9: "alpha within [0, ALPHA_MAX] on every evaluation pass",
              10: "C1 receives no audio; shuffle controls (C2 / D1sh / D2sh) permute audio within each split only, "
                  "labels and IMU untouched",
              11: "audio and IMU z-score statistics from training windows only",
              12: "B0 and R1 per-fold F1 within 0.06 of E18b's (SOFT)",
              14: "strong/weak sets from fit windows only; disjoint; complete; D1r sizes equal to D1's",
              15: "wavelet scattering: finite; sinusoid activates the nearest first-order path; window-grid alignment"}


def hook(n, ok, msg, stop=None):
    ok = bool(ok)
    HOOKS[n].append((ok, msg))
    print(f"[HOOK {n}] {'PASS' if ok else 'FAIL'} — {msg}")
    stop = (n <= 4) if stop is None else stop
    if not ok and stop:
        raise RuntimeError(f"HOOK {n} FAILED: {msg}. Stopping before anything is trained.")
    return ok


# ---- figures (palette: validated categorical order; sequential = one hue; diverging = two hues + neutral midpoint) ----
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CLASS_MARKERS = ["o", "s", "^", "D", "v"]
DAY_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
DAY_MARKERS = ["o", "s", "^", "D", "v"]
SEQ_CMAP, DIV_CMAP = "Blues", "RdBu_r"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
                     "grid.linewidth": 0.6, "lines.linewidth": 1.5, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})
FREQ_LABEL = "per-channel FFT bin (1-128)"
HZ_LABEL = "approx. Hz if the per-channel rate were 16 kHz (unverified)"
FREQ_CAVEAT = ("Frequency axis = per-channel FFT bin of the de-interleaved channels. The Hz axis assumes a per-channel "
               "rate of 16 kHz, which is UNVERIFIED (the files carry no sample rate).")
FIGURES = []


def add_bin_axis(ax, axis="x"):
    """Secondary Hz axis for a per-channel-bin axis. Label only: HZ_PER_BIN is never used in a computation."""
    fwd, inv = (lambda b: b * HZ_PER_BIN), (lambda h: h / HZ_PER_BIN)
    if axis == "x":
        sec = ax.secondary_xaxis("top", functions=(fwd, inv))
        sec.set_xlabel(HZ_LABEL, fontsize=7)
    else:
        sec = ax.secondary_yaxis("right", functions=(fwd, inv))
        sec.set_ylabel(HZ_LABEL, fontsize=7)
    sec.tick_params(labelsize=7)
    ax._bin_caveat = True
    return sec


def save_fig(fig, name, caveat, freq_axes=()):
    """Every figure is saved as PNG with its caveat printed under it. Frequency figures must carry the Hz caveat."""
    assert caveat, f"{name}: every figure needs a caveat"
    freq_ok = all(getattr(ax, "_bin_caveat", False) for ax in freq_axes) and (not freq_axes or "UNVERIFIED" in caveat)
    fig.text(0.5, -0.01, caveat, ha="center", va="top", fontsize=8, color="#52514e", wrap=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    FIGURES.append({"name": name, "freq": bool(freq_axes), "freq_caveat_ok": freq_ok})
    print(f"  saved {FIG_DIR / (name + '.png')}" + ("" if freq_ok else "  <- MISSING frequency caveat"))


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

In [ ]:
# ==============================================================================
# 0f. MEMORY / DISK LOGGING + RESTORE finished runs from an attached earlier output (RESTORE_FROM)
# ==============================================================================
def mem_report(tag):
    """Process RSS, available RAM, GPU memory and free disk in /kaggle/working; flushed so a crash cannot swallow it."""
    rss = avail = float("nan")
    try:
        with open("/proc/self/status") as fh:
            rss = next(int(l.split()[1]) for l in fh if l.startswith("VmRSS")) / 1e6
        with open("/proc/meminfo") as fh:
            avail = next(int(l.split()[1]) for l in fh if l.startswith("MemAvailable")) / 1e6
    except Exception:
        pass
    gpu = (f"GPU alloc {torch.cuda.memory_allocated() / 1e9:.2f} GB, reserved {torch.cuda.memory_reserved() / 1e9:.2f} GB"
           if torch.cuda.is_available() else "no GPU")
    disk = shutil.disk_usage(OUT).free / 1e9 if Path(OUT).exists() else float("nan")
    print(f"[MEM] {tag}: process RSS {rss:.2f} GB | RAM available {avail:.2f} GB | {gpu} | disk free {disk:.1f} GB",
          flush=True)


import shutil
mem_report("after setup")


def _find_results_csv(root):
    """Bounded search (depth <= 3) for the results CSV; never walks dataset clones or caches."""
    for depth in range(4):
        hits = sorted(Path(root).glob("/".join(["*"] * depth + [RESULTS_CSV.name])))
        if hits:
            return hits[0]
    return None


_roots = [] if RESTORE_FROM is None else ([RESTORE_FROM] if isinstance(RESTORE_FROM, (str, Path)) else list(RESTORE_FROM))
for _root in _roots:
    print(f"RESTORE: searching {_root} (depth <= 3) for {RESULTS_CSV.name} ...", flush=True)
    try:
        if not Path(_root).is_dir():
            loud(f"RESTORE_FROM {_root} is not a folder. Nothing restored from it.")
            continue
        _csv = _find_results_csv(_root)
        if _csv is None:
            loud(f"RESTORE: no {RESULTS_CSV.name} within depth 3 of {_root}. Nothing restored from it.")
            continue
        _src = _csv.parent
        _old = pd.read_csv(_csv).drop_duplicates(["arm", "fold", "seed"], keep="first")
        print(f"RESTORE: found {_csv} ({len(_old)} rows; seeds {sorted(set(_old.seed.astype(int)))})", flush=True)
        if not set(_old.seed.astype(int)) <= set(SEEDS):
            loud(f"{_csv} contains seeds {sorted(set(_old.seed.astype(int)))} that are not E23 seeds {SEEDS}. "
                 "Nothing restored (it is not an E23 output).")
            continue
        _jobs = []
        for r in _old.itertuples():
            _nm = f"{r.arm}_{int(r.fold)}_{int(r.seed)}"
            _jobs.append((_src / PROBS_DIR.name / f"{_nm}.npz", PROBS_DIR / f"{_nm}.npz"))
            if r.arm in CKPT_ARMS:
                _jobs.append((_src / CKPT_DIR.name / f"{_nm}.pt", CKPT_DIR / f"{_nm}.pt"))
        _copied, _present, _missing = 0, 0, []
        for i, (_a, _b) in enumerate(_jobs, 1):
            if _b.exists():
                _present += 1
            elif _a.exists():
                shutil.copyfile(_a, _b)
                _copied += 1
            else:
                _missing.append(_a.name)
            if i % 50 == 0 or i == len(_jobs):
                print(f"RESTORE: {i}/{len(_jobs)} files checked | copied {_copied} | already present {_present} | "
                      f"missing {len(_missing)}", flush=True)
        if _missing:
            print(f"RESTORE: {len(_missing)} listed files are missing in the source (those runs will train again), "
                  f"e.g. {_missing[:5]}", flush=True)
        _merged = pd.concat([pd.read_csv(RESULTS_CSV), _old]) if RESULTS_CSV.exists() else _old
        _merged.drop_duplicates(["arm", "fold", "seed"], keep="first").to_csv(RESULTS_CSV, index=False)
        print(f"RESTORED from {_src}: {len(_old)} result rows, {_copied} files copied into {OUT}.", flush=True)
    except Exception as exc:
        loud(f"RESTORE from {_root} FAILED: {exc!r}\nNothing more is restored from it; runs without files will train.")
mem_report("after restore")

## 1. Data: parse once, then ONE set of paired windows (v29 cells 3a–3c, verbatim; hooks renumbered)
- **3a — parse.**
  - IMU: the IMU notebook's schema-first parser, **file order, never sorted**.
  - Audio: the audio notebook's per-chunk 80 R0 features (`pow[:, :, 1:17] ** 0.25`) plus `recording_started_time`.
- **3b — metadata.** v24's corrected audio anchor, then the exclusion rule computed from metadata (offset / duration
  ≥ 0.5 → excluded; it must equal the 4 known names), then the counts.
- **3c — paired windows.**
  - Audio (64 × 80) and IMU (128 × 6) come from the **same real time span of the same mission**. The window grid
    starts at audio onset with a 1.28 s step.
  - IMU: 128 real file-order samples chosen evenly by index.
  - Audio: 64 chunks chosen evenly by index among the chunks inside the span.
  - Includes a hash rebuild of 25 sampled windows.

In [ ]:
# ==============================================================================
# 3a. PARSE (all 99 folders: the 4 exclusions are computed from metadata, not assumed) -> v29_cache/<mission>.npz
# ==============================================================================
def unwrap_message(obj):                                   # IMU notebook, unchanged
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def get_vec(d, candidates):                                # IMU notebook, unchanged
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        vals, ok = [], True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None


def parse_imu_file(path):
    """IMU notebook's parser: rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER."""
    rows, bad = [], 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, ["accelerometer_m_s2", "accelerometer_m_s2[{i}]", "accelerometer", "acceleration"])
                gyro = get_vec(obj, ["gyro_rad", "gyro_rad[{i}]", "gyro", "angular_velocity"])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1
    return np.asarray(rows, dtype=np.float64).reshape(-1, 7), bad


def parse_mission(mdir):
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        z = np.load(cache)
        return {k: z[k] for k in z.files}
    s_path = sorted(mdir.glob("*SensorCombined*.jsonl"))[0]
    a_path = sorted(mdir.glob("*AudioBuffer*.jsonl"))[0]
    imu, bad = parse_imu_file(s_path)
    chunks, lens, rst, rsts = [], Counter(), None, set()
    with open(a_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            x = rec.get("data")
            if x is None or len(x) == 0:
                continue
            if "recording_started_time" in rec:
                rsts.add(float(rec["recording_started_time"]))
                if rst is None:
                    rst = float(rec["recording_started_time"])          # v24: the FIRST record's value
            lens[len(x)] += 1
            chunks.append(x)
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal}"
    assert rst is not None, f"{mdir.name}: no recording_started_time — the audio cannot be anchored"
    keep_idx = np.array([i for i, x in enumerate(chunks) if len(x) == modal], dtype=np.int64)
    raw = np.asarray([chunks[i] for i in keep_idx], dtype=np.int64)
    n_all = len(chunks)
    del chunks
    frames = raw.reshape(len(raw), FRAME, N_SLOTS)                # sample i -> frame i // 6, slot i % 6
    slot6_zero = bool(np.all(frames[:, :, DROP_SLOT] == 0))
    ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
    ch = ch - ch.mean(axis=1, keepdims=True)                      # per-chunk mean removed, per channel
    pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)    # (n, 5, 129)
    feats = (pw[:, :, KEEP_BINS] ** 0.25).reshape(len(pw), N_AUDIO_FEAT).astype(np.float32)   # sqrt|rFFT|, channel-major
    out = {"imu_t": imu[:, 0], "imu_x": imu[:, 1:].astype(np.float32), "imu_bad": bad, "feats": feats,
           "keep_idx": keep_idx, "n_all_chunks": n_all, "rst": rst, "n_rst_values": len(rsts), "slot6_zero": slot6_zero}
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, cache)
    return out


def mission_day(name):
    d = name[8:18].replace("_", "-")
    return DAY_MERGE.get(d, d)


ALL_DIRS, LABEL_OF = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not (class_dir.is_dir() and class_dir.name.isdigit()):
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF[mdir.name] = int(class_dir.name)
_t0 = time.time()
PARSED = {}
for i, mdir in enumerate(ALL_DIRS, 1):
    PARSED[mdir.name] = parse_mission(mdir)
    if i % 10 == 0 or i == len(ALL_DIRS):
        print(f"  parsed {i}/{len(ALL_DIRS)} missions | {time.time() - _t0:.0f} s")

In [ ]:
# ==============================================================================
# 3b. METADATA — v24's corrected audio anchor, exclusion rule, mission / class / day counts (HOOK data)
# ==============================================================================
def timebase(P):
    """v24 CORRECTED_ALIGNMENT time axes. IMU: FILE ORDER, seconds from the first sample. Audio: anchored at
    recording_started_time; chunk duration = (mission duration - offset) / number of chunks."""
    t_us = P["imu_t"]
    s_times = (t_us - t_us[0]) / 1e6
    duration = float(s_times[-1] - s_times[0])
    offset = (float(P["rst"]) - float(t_us[0])) / 1e6
    chunk = (duration - offset) / int(P["n_all_chunks"])
    a_times = (offset + np.arange(int(P["n_all_chunks"]), dtype=np.float64) * chunk)[P["keep_idx"]]
    return s_times, a_times, duration, offset, chunk


META = []
for name, P in PARSED.items():
    s_times, a_times, dur, off, ch = timebase(P)
    META.append({"mission": name, "class": LABEL_OF[name], "day": mission_day(name), "duration_s": dur, "offset_s": off,
                 "ratio": off / dur, "chunk_ms": 1e3 * ch, "n_imu": len(s_times), "imu_inversions": int((np.diff(s_times) < 0).sum()),
                 "n_chunks": int(P["n_all_chunks"]), "chunks_dropped": int(P["n_all_chunks"]) - len(P["keep_idx"]),
                 "rst_values": int(P["n_rst_values"]), "slot6_zero": bool(P["slot6_zero"])})
META = pd.DataFrame(META).set_index("mission")
EXCLUDED = set(META.index[META.ratio >= EXCLUDE_RATIO])
KEPT = sorted(set(META.index) - EXCLUDED)
M = META.loc[KEPT]
LABEL = {n: int(M.loc[n, "class"]) for n in KEPT}
DAY = {n: M.loc[n, "day"] for n in KEPT}
_cls = np.bincount(M["class"], minlength=N_CLASSES).tolist()
_days = [int((M.day == d).sum()) for d in DAYS]
print(META.loc[sorted(EXCLUDED), ["class", "day", "offset_s", "duration_s", "ratio"]].round(3).to_string())
print(f"\nKept {len(KEPT)} missions | max kept ratio {M.ratio.max():.3f} | median offset {M.offset_s.median():.2f} s | "
      f"chunk duration {M.chunk_ms.median():.2f} ms (median; v24's corrected-anchor formula) | "
      f"non-modal chunks dropped {int(M.chunks_dropped.sum())} | missions with IMU timestamp inversions "
      f"{int((M.imu_inversions > 0).sum())} (total {int(M.imu_inversions.sum())})")
print(pd.crosstab(M.day, M["class"]).reindex(DAYS).to_string())
hook(1, EXCLUDED == EXPECTED_EXCLUDED, f"exclusion set == the 4 known names: {sorted(EXCLUDED)}")
hook(1, len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
     f"{len(KEPT)} missions, classes {_cls}, days {_days} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, {EXPECTED_DAY_COUNTS})")
hook(1, bool(M.slot6_zero.all()) and bool((M.rst_values == 1).all()),
     f"audio slot 6 identically zero in {int(M.slot6_zero.sum())}/{len(M)}; recording_started_time constant in "
     f"{int((M.rst_values == 1).sum())}/{len(M)}")

In [ ]:
# ==============================================================================
# 3c. ONE SET OF PAIRED WINDOWS (v24 CORRECTED_ALIGNMENT): audio (64 x 80) and IMU (128 x 6, file order) from the SAME
#     real time span of the SAME mission. Every model below reads these arrays (unimodal models ignore one half).
# ==============================================================================
def paired_windows(name, P):
    s_times, a_times, dur, _, _ = timebase(P)
    tel, feats = P["imu_x"], P["feats"]
    max_time = min(s_times[-1], a_times[-1])
    out, i = [], 0
    t_start = max(0.0, float(a_times[0]))                  # v24: CORRECTED_WINDOW_START = "audio_onset"
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue
        s_idx = np.where(s_mask)[0]
        pick = s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]
        tau_s_abs = s_times[pick]
        x_s = tel[pick].astype(np.float32)                 # real file-order samples (IMU notebook rule)
        x_s_v24 = np.stack([np.interp(tau_s_abs, s_times[s_mask], tel[s_mask, c]) for c in range(N_SENSOR_CHANNELS)],
                           axis=1).astype(np.float32)      # v24's np.interp form, kept only for the comparison count
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            a_pick = a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]
            tau_a_abs = a_times[a_pick]
            x_a = feats[a_pick].astype(np.float32)         # == v24's np.interp at exact (strictly increasing) chunk times
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
            x_a = np.stack([np.interp(tau_a_abs, a_times, feats[:, c], left=0.0, right=0.0) for c in range(N_AUDIO_FEAT)],
                           axis=1).astype(np.float32)
        rel = (t_start + WINDOW_SEC / 2) / dur
        out.append({"window_id": f"{name}#{i:04d}", "mission": name, "cls": LABEL[name], "day": DAY[name],
                    "t_start": t_start, "t_end": t_end,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady"),
                    "x_a": x_a, "x_s": x_s, "tau_a": ((tau_a_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "tau_s": ((tau_s_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "v24_interp_differs": bool(not np.array_equal(x_s, x_s_v24)),
                    "v24_interp_maxdiff": float(np.abs(x_s - x_s_v24).max())})
        i += 1
        t_start += STEP_SEC                                 # v24: accumulated float step
    return out


_W = []
for name in KEPT:
    _W.extend(paired_windows(name, PARSED[name]))
WMETA = pd.DataFrame([{k: v for k, v in w.items() if k not in ("x_a", "x_s", "tau_a", "tau_s")} for w in _W])
XA = np.stack([w["x_a"] for w in _W])                      # (N, 64, 80)
XS = np.stack([w["x_s"] for w in _W])                      # (N, 128, 6)
TAU_A = np.stack([w["tau_a"] for w in _W])                 # (N, 64)   in window units (v24 dataset: tau / window_sec)
TAU_S = np.stack([w["tau_s"] for w in _W])                 # (N, 128)
YW = WMETA.cls.values.astype(np.int64)
del _W
gc.collect()
print(f"Paired windows: {len(WMETA)} | XA {XA.shape} | XS {XS.shape} | per class {np.bincount(YW, minlength=N_CLASSES).tolist()} | "
      f"per phase {WMETA.phase.value_counts().to_dict()}")
print(f"IMU: windows where the file-order direct pick differs from v24's np.interp form: "
      f"{int(WMETA.v24_interp_differs.sum())} / {len(WMETA)} (max abs diff {WMETA.v24_interp_maxdiff.max():.4g}) — "
      "nonzero only where a window's IMU timestamps go backwards")

# ---- pairing hooks ----
_span_ok = bool((TAU_A >= -1e-6).all() and (TAU_A <= 1 + 1e-6).all() and (TAU_S >= -1e-6).all() and (TAU_S <= 1 + 1e-6).all())
hook(2, _span_ok and WMETA.window_id.is_unique,
     "every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission "
     "(both come from a single record: same mission id, same span)")
_rng_h = np.random.default_rng(0)
_sample = np.sort(_rng_h.choice(len(WMETA), size=min(N_HASH_SAMPLE, len(WMETA)), replace=False))
_hash_ok = True
for j in _sample:
    r = WMETA.iloc[j]
    rebuilt = {w["window_id"]: w for w in paired_windows(r.mission, PARSED[r.mission])}[r.window_id]
    same = sha16(rebuilt["x_a"], rebuilt["x_s"]) == sha16(XA[j], XS[j]) and abs(rebuilt["t_start"] - r.t_start) < 1e-12
    _hash_ok &= same
hook(2, _hash_ok, f"hash check on {len(_sample)} sampled windows: stored (audio, IMU) == rebuilt from the raw "
                          f"mission at the recorded (mission, t_start)")
_fin = bool(np.isfinite(XA).all() and np.isfinite(XS).all())
hook(2, _fin and XA.shape[1:] == (TA_AUDIO, N_AUDIO_FEAT) and XS.shape[1:] == (TS_SENSOR, N_SENSOR_CHANNELS),
     f"shapes audio {XA.shape[1:]}, IMU {XS.shape[1:]}; all finite: {_fin}")
PAIRED_SHA = sha16(XA, XS)
print(f"Paired-window content sha1: {PAIRED_SHA}")

In [ ]:
# ==============================================================================
# 1d. PAIRING COVERAGE (HOOK 4) + compatibility names for the E19 fold cell
# ==============================================================================
_rows = []
for name in KEPT:
    s_times, a_times, dur, off, ch = timebase(PARSED[name])
    w = WMETA[WMETA.mission == name]
    n_real = np.array([int(((a_times >= a) & (a_times <= b)).sum()) for a, b in zip(w.t_start, w.t_end)])
    # IMU reference grid (IMU notebook: 2.56 s windows, 1.28 s stride from the first file-order sample)
    imu_starts = np.arange(0, s_times[-1] - WIN_S + 1e-9, STRIDE_S)
    covered = np.array([bool(len(w) and np.min(np.abs(w.t_start.values - s)) <= STEP_SEC / 2 + 1e-9) for s in imu_starts])
    _rows.append({"mission": name, "paired_windows": len(w), "with_real_audio": int((n_real >= 2).sum()),
                  "median_chunks_per_window": float(np.median(n_real)) if len(n_real) else 0.0,
                  "imu_grid_windows": len(imu_starts), "imu_grid_covered": int(covered.sum()), "audio_offset_s": off})
PAIR_COV = pd.DataFrame(_rows).set_index("mission")
PAIR_COV.to_csv(OUT / "e23_pairing_coverage.csv")
REAL_AUDIO_SHARE = PAIR_COV.with_real_audio.sum() / PAIR_COV.paired_windows.sum()
IMU_GRID_COVERAGE = PAIR_COV.imu_grid_covered.sum() / PAIR_COV.imu_grid_windows.sum()
print(f"Paired windows {int(PAIR_COV.paired_windows.sum())} | with >= 2 real audio chunks: {100 * REAL_AUDIO_SHARE:.2f}% | "
      f"median audio chunks per window {PAIR_COV.median_chunks_per_window.median():.0f} (64 are picked by index)")
print(f"IMU-reference grid windows {int(PAIR_COV.imu_grid_windows.sum())} | covered by a paired window (>= 75% overlap): "
      f"{100 * IMU_GRID_COVERAGE:.1f}% | median audio offset {PAIR_COV.audio_offset_s.median():.1f} s (the IMU-only "
      f"stretch before audio onset has no audio and is NOT in any E18 arm)")
hook(4, REAL_AUDIO_SHARE >= PAIRING_MIN_COVERAGE,
     f"{100 * REAL_AUDIO_SHARE:.2f}% of paired windows carry real audio chunks (>= {100 * PAIRING_MIN_COVERAGE:.0f}% "
     f"required; the rest would be v29's zero-filled interpolation). Pairing method: v29 paired_windows "
     f"(v24 CORRECTED_ALIGNMENT, same span of the same mission).")
if IMU_GRID_COVERAGE < PAIRING_MIN_COVERAGE:
    print(f"WARN (soft): only {100 * IMU_GRID_COVERAGE:.1f}% of the IMU-only reference windows have a paired counterpart: "
          "audio starts after the IMU log. E18 compares arms on the paired windows only (B0 is retrained there), so the "
          "comparison stays paired; the absolute B0 score is not the unpaired IMU reference's.")

# ---- names used by the E19 fold cell (verbatim below) ----
missions = pd.DataFrame({"mission": KEPT, "class": [LABEL[n] for n in KEPT], "day": [DAY[n] for n in KEPT]})
imu_data = {n: {"day": DAY[n], "class": LABEL[n]} for n in KEPT}
groups = WMETA.mission.values
MISSION_DAY = {n: DAY[n] for n in KEPT}
MISSION_CLASS = {n: LABEL[n] for n in KEPT}

## 2. Day-out folds (E19 template = IMU reference fold cell, verbatim) and split hook 3
- Fold k tests on day k.
- Inner validation = v27's `_split_inner_val` if `minor-project-v27.ipynb` is attached under `/kaggle/input`;
  otherwise the fixed seed-2026 fallback. **The rule that ran is printed.**

In [ ]:
# ==============================================================================
# DAY-OUT FOLDS — identical to the audio notebook (fold k = DAYS[k]; v27 inner-validation rule, else seed 2026)
# ==============================================================================
def extract_function(nb_path,fname):
    try:
        nb=json.loads(Path(nb_path).read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"  could not read {nb_path}: {exc!r}")
        return None
    for cell in nb.get("cells",[]):
        if cell.get("cell_type")!="code":
            continue
        src=cell["source"] if isinstance(cell["source"],str) else "".join(cell["source"])
        src="\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!","%")))
        try:
            tree=ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name==fname:
                return ast.get_source_segment(src,node)
    return None


LABEL={Path(m):int(c) for m,c in zip(missions["mission"],missions["class"])}   # v27's rule keys missions by Path (.name)
_p27=find_file("minor-project-v27.ipynb",["version 27","Version 27"])
_src27=extract_function(_p27,"_split_inner_val") if _p27 else None
if _src27 is not None and hashlib.sha1(_src27.encode()).hexdigest()[:16]==V27_INNER_VAL_SHA1:
    _ns27={"np":np,"N_CLASSES":N_CLASSES,"LABEL":LABEL,"INNER_VAL_SEED":INNER_VAL_SEED,"INNER_VAL_FRACTION":INNER_VAL_FRACTION}
    exec(_src27,_ns27)
    inner_val=lambda pool,k: _ns27["_split_inner_val"](pool,0,k)
    INNER_VAL_RULE=f"v27 _split_inner_val (from {_p27}; seed [{INNER_VAL_SEED}, 0, fold]) — same as the audio notebook"
else:
    loud(("v27 notebook not found" if _p27 is None else f"v27 _split_inner_val unreadable/fingerprint mismatch at {_p27}")
         + f"\nFALLBACK inner validation: per-class ~15% (min 1) mission hold-out, fixed seed {INNER_VAL_FALLBACK_SEED} "
           "(the audio notebook's fallback, identical code).")

    def inner_val(pool,k):
        rng=np.random.default_rng([INNER_VAL_FALLBACK_SEED,k])
        train,val=[],[]
        for c in range(N_CLASSES):
            ms=sorted((m for m in pool if LABEL[m]==c),key=lambda m:m.name)
            perm=rng.permutation(len(ms))
            n_val=max(1,int(round(INNER_VAL_FRACTION*len(ms))))
            val+=[ms[i] for i in perm[:n_val]]
            train+=[ms[i] for i in perm[n_val:]]
        return train,val

    INNER_VAL_RULE=f"FALLBACK per-class hold-out, seed {INNER_VAL_FALLBACK_SEED} (same as the audio notebook's fallback)"
print("Inner-validation rule:",INNER_VAL_RULE)

FOLDS={}
_all=set(missions["mission"])
for k,day in enumerate(DAYS):
    test=sorted(missions.loc[missions["day"]==day,"mission"])
    tr,va=inner_val([Path(m) for m in sorted(_all-set(test))],k)
    FOLDS[k]={"day":day,"train":sorted(p.name for p in tr),"val":sorted(p.name for p in va),"test":test}
    f=FOLDS[k]
    assert not (set(f["train"])&set(f["val"])) and not (set(f["train"]+f["val"])&set(test))
    assert set(f["train"]+f["val"]+test)==_all
    assert day not in {imu_data[m]["day"] for m in f["train"]+f["val"]}, f"fold {k}: test day in training"
assert sorted(m for f in FOLDS.values() for m in f["test"])==sorted(_all)

print(f"{'fold':<5}{'test day':<12}{'missions tr/val/test':<22}{'test missions per class':<26}{'windows tr/val/test'}")
for k,f in FOLDS.items():
    per_class=np.bincount([imu_data[m]["class"] for m in f["test"]],minlength=N_CLASSES).tolist()
    n_w="/".join(str(int(np.isin(groups,f[s]).sum())) for s in ("train","val","test"))
    print(f"{k:<5}{f['day']:<12}{'/'.join(str(len(f[s])) for s in ('train','val','test')):<22}{str(per_class):<26}{n_w}"
          + ("   <- SMALL fold (10 missions)" if f["day"]==SMALL_FOLD_DAY else ""))

In [ ]:
# ==============================================================================
# 3b. SPLIT HOOK 3: no test-day mission in train/val; inner validation has all 5 classes; per-fold mission hashes
# ==============================================================================
_ok3 = True
for k, f in FOLDS.items():
    _ok3 &= f["day"] not in {MISSION_DAY[m] for m in f["train"] + f["val"]}
    _ok3 &= not (set(f["train"] + f["val"]) & set(f["test"])) and not (set(f["train"]) & set(f["val"]))
    _ok3 &= {MISSION_CLASS[m] for m in f["val"]} == set(range(N_CLASSES))
    _ok3 &= {MISSION_CLASS[m] for m in f["train"]} == set(range(N_CLASSES))
FOLD_HASH = {k: hashlib.sha1(json.dumps({s: f[s] for s in ("train", "val", "test")}).encode()).hexdigest()[:12]
             for k, f in FOLDS.items()}
hook(3, _ok3, "every fold: test day absent from train/val, no mission in two sets, inner validation and training "
              f"contain all 5 classes | rule: {INNER_VAL_RULE} | fold hashes {FOLD_HASH}")

## 3. Models (E19 template: AudioMamba + IMUMamba, references verbatim; real mamba_ssm)
The models are unchanged. Fusion arms read each model's pooled embedding (the input of its `classifier`) through a
forward pre-hook.

In [ ]:
# ==============================================================================
# 6a. MODELS: AudioMamba (audio reference) + IMUMamba (IMU reference) — real mamba_ssm (HOOK 6)
# ==============================================================================
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda")

# E19: the v28 notebook is not read; the reference's transcribed settings are used unchanged.


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM (v28)."""

    def __init__(self):
        super().__init__()
        self.norm = nn.LayerNorm(D_MODEL)
        self.mamba = MAMBA_CLASS(d_model=D_MODEL, d_state=MAMBA_D_STATE, d_conv=MAMBA_D_CONV, expand=MAMBA_EXPAND)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


class AudioMamba(nn.Module):
    """v28 UnimodalMamba, audio path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention
    pooling -> classifier."""

    def __init__(self, in_dim=N_FEAT):
        super().__init__()
        self.audio_encoder = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, x):
        h = self.token_norm(self.audio_encoder(x) + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1))


def verify_model(model):
    real = all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks) and MAMBA_CLASS.__module__.startswith("mamba_ssm")
    blocks_ok = len(model.mamba_blocks) == MAMBA_NUM_BLOCKS and all(
        (b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND) for b in model.mamba_blocks)
    n_mamba = sum(p.numel() for p in model.mamba_blocks.parameters())
    n_params = sum(p.numel() for p in model.parameters())
    return real and blocks_ok and n_mamba == EXPECTED_MAMBA_PARAMS, n_params, n_mamba


set_all_seeds(SEEDS[0])
_probe = AudioMamba().to(device)
_ok, N_PARAMS, _n_mamba = verify_model(_probe)
assert _ok, "model is not the expected real-mamba_ssm architecture"
print(f"AudioMamba: {N_PARAMS:,} parameters (Mamba blocks {_n_mamba:,} == {EXPECTED_MAMBA_PARAMS:,}) | block class "
      f"{MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} | modules {[n for n, _ in _probe.named_children()]}")
with torch.no_grad():
    assert _probe(torch.zeros(2, TA, N_FEAT, device=device)).shape == (2, N_CLASSES)
hook(6, _ok, f"every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); {N_PARAMS:,} parameters",
     stop=False)
del _probe
torch.cuda.empty_cache()


# ---- IMUMamba: IMU reference, verbatim (it uses the same MambaBlock wrapper around mamba_ssm.Mamba) ----
class IMUMamba(nn.Module):
    """v28 UnimodalMamba, sensor path (S1)."""
    def __init__(self, in_dim=6, n_classes=N_CLASSES):
        super().__init__()
        self.sensor_encoder=nn.Sequential(
            nn.Conv1d(in_dim,64,kernel_size=5,padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64,D_MODEL,kernel_size=3,padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding=nn.Embedding(1,D_MODEL)
        self.token_norm=nn.LayerNorm(D_MODEL)
        self.mamba_blocks=nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn=nn.Sequential(nn.Linear(D_MODEL,64),nn.Tanh(),nn.Linear(64,1))
        self.classifier=nn.Sequential(nn.Dropout(DROPOUT),nn.Linear(D_MODEL,D_MODEL//2),nn.GELU(),
                                      nn.Linear(D_MODEL//2,n_classes))

    def forward(self,x):
        z=self.sensor_encoder(x.transpose(1,2)).transpose(1,2)
        h=self.token_norm(z+self.modality_embedding.weight.view(1,1,-1))
        for b in self.mamba_blocks:
            h=b(h)
        a=F.softmax(self.pool_attn(h),dim=1)
        return self.classifier((h*a).sum(dim=1))


DEVICE = device                                                  # the IMU reference's name for the same device
set_all_seeds(SEEDS[0])
_probe = IMUMamba().to(DEVICE)
_ok_i, N_PARAMS_IMU, _n_mamba_i = verify_model(_probe)
with torch.no_grad():
    _shape_i = _probe.eval()(torch.zeros(2, TS, len(CHANNELS), device=DEVICE)).shape == (2, N_CLASSES)
hook(6, _ok_i and _shape_i, f"IMUMamba: every block is mamba_ssm.Mamba (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}); "
                            f"{N_PARAMS_IMU:,} parameters (Mamba {_n_mamba_i:,})", stop=True)
N_PARAMS_AUDIO = N_PARAMS
del _probe
torch.cuda.empty_cache()

## 4. Training core (E18b, verbatim)
B0, C1 and R1 are trained by E18b's `run_arm`, unchanged. The dual-path arms reuse the same pieces:
`train_loop`, `predict`, `val_f1`, `Pooled`, `state_hash`, `c2_permutation`, `mission_level` and the
train-only z-score.

In [ ]:
# ==============================================================================
# 4a. E18 TRAINING CORE (new code; B0/AUD == the reference training procedure)
# ==============================================================================
# mission aggregation: audio reference cell 6b, verbatim
def mission_level(missions, y, P):
    """Mission prediction = argmax of the mean window probabilities."""
    df = pd.DataFrame(P, columns=range(N_CLASSES))
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g.y.first().values, g[list(range(N_CLASSES))].mean().values, g.y.first().index.values


XA_G = torch.from_numpy(XA).to(DEVICE)                    # paired windows on the GPU: (N, 64, 80) and (N, 128, 6)
XS_G = torch.from_numpy(XS).to(DEVICE)
Y_G = torch.from_numpy(YW).to(DEVICE)
Y_HASH = sha16(YW)
WID, WMIS, WDAY, WPHASE = WMETA.window_id.values, WMETA.mission.values, WMETA.day.values, WMETA.phase.values


def fold_prep(k):
    """v29 FOLD_PREP rule: indices, per-feature audio / per-channel IMU z-score from TRAINING windows only, class weights."""
    f = FOLDS[k]
    idx = {s: np.where(WMETA.mission.isin(f[s]).values)[0] for s in ("train", "val", "test")}
    assert set(WMETA.mission.values[idx["train"]]) == set(f["train"])
    a_tr = XA[idx["train"]].reshape(-1, N_AUDIO_FEAT)
    s_tr = XS[idx["train"]].reshape(-1, N_SENSOR_CHANNELS)
    a_mu, a_sd = a_tr.mean(0, dtype=np.float64).astype(np.float32), (a_tr.std(0, dtype=np.float64) + AUDIO_STD_EPS).astype(np.float32)
    s_mu, s_sd = s_tr.mean(0, dtype=np.float64).astype(np.float32), (s_tr.std(0, dtype=np.float64) + SENSOR_STD_EPS).astype(np.float32)
    del a_tr, s_tr
    counts = np.bincount(YW[idx["train"]], minlength=N_CLASSES)
    return {"idx": idx, "a_mu": a_mu, "a_sd": a_sd, "s_mu": s_mu, "s_sd": s_sd, "zhash": sha16(a_mu, a_sd, s_mu, s_sd),
            "class_weights": (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)}


FOLD_PREP = {k: fold_prep(k) for k in FOLDS}
hook(11, all(set(WMIS[FOLD_PREP[k]["idx"]["train"]]) == set(FOLDS[k]["train"]) for k in FOLDS),
     "audio and IMU z-score statistics computed from each fold's TRAINING windows only: "
     + ", ".join(f"fold {k} {FOLD_PREP[k]['zhash']}" for k in FOLDS), stop=True)


def zscored(k):
    P = FOLD_PREP[k]
    t = lambda a: torch.from_numpy(a).to(DEVICE)
    return (XA_G - t(P["a_mu"])) / t(P["a_sd"]), (XS_G - t(P["s_mu"])) / t(P["s_sd"])


class Pooled(nn.Module):
    """Wraps an UNCHANGED reference model; forward returns (pooled embedding = the classifier's input, logits)."""

    def __init__(self, model):
        super().__init__()
        self.model = model
        self._z = None
        self.model.classifier.register_forward_pre_hook(self._grab)

    def _grab(self, module, inp):
        self._z = inp[0]

    def forward(self, x):
        logits = self.model(x)
        return self._z, logits


class JointFusion(nn.Module):
    """B2: IMU and audio encoders end-to-end, concatenated pooled embeddings -> linear head (CE only)."""

    def __init__(self):
        super().__init__()
        self.imu = Pooled(IMUMamba())
        self.aud = Pooled(AudioMamba())
        self.head = nn.Linear(2 * D_MODEL, N_CLASSES)

    def forward(self, xa, xs):
        hi, _ = self.imu(xs)
        ha, _ = self.aud(xa)
        return self.head(torch.cat([hi, ha], dim=1))


class ResidualFusion(nn.Module):
    """z_final = z_IMU + alpha * dz; dz = W2 relu(W1 h) with W2, b2 ZERO-initialised; alpha = ALPHA_MAX * sigmoid(g(h))."""

    def __init__(self, anchor, use_audio):
        super().__init__()
        self.anchor = Pooled(anchor)
        self.aud = Pooled(AudioMamba()) if use_audio else None
        d_in = 2 * D_MODEL if use_audio else D_MODEL
        self.w1 = nn.Linear(d_in, R_HIDDEN)
        self.w2 = nn.Linear(R_HIDDEN, N_CLASSES)
        nn.init.zeros_(self.w2.weight)
        nn.init.zeros_(self.w2.bias)
        self.gate = nn.Linear(d_in, 1)
        self.anchor_frozen = True
        self.set_anchor_frozen(True)

    def set_anchor_frozen(self, frozen):
        self.anchor_frozen = frozen
        for p in self.anchor.parameters():
            p.requires_grad_(not frozen)
        if frozen:
            self.anchor.eval()

    def train(self, mode=True):
        super().train(mode)
        if self.anchor_frozen:
            self.anchor.eval()                        # a frozen anchor never updates BatchNorm statistics
        return self

    def forward(self, xa, xs):
        if self.anchor_frozen:
            with torch.no_grad():
                hi, zi = self.anchor(xs)
        else:
            hi, zi = self.anchor(xs)
        h = torch.cat([hi, self.aud(xa)[0]], dim=1) if self.aud is not None else hi
        dz = self.w2(F.relu(self.w1(h)))
        alpha = ALPHA_MAX * torch.sigmoid(self.gate(h))
        return zi + alpha * dz, zi, alpha, dz


def state_hash(module):
    return hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                 for v in module.state_dict().values())).hexdigest()[:16]


def c2_permutation(k, seed):
    """Audio window permutation WITHIN each split (fit / val / test); identity elsewhere. Labels are never touched."""
    rng = np.random.default_rng([C2_PERM_SEED, k, seed])
    perm = np.arange(len(WID))
    shares = {}
    for s, ix in FOLD_PREP[k]["idx"].items():
        perm[ix] = ix[rng.permutation(len(ix))]
        assert set(perm[ix]) == set(ix), f"C2: permutation leaves split {s}"
        shares[s] = float((perm[ix] != ix).mean())
    return perm, shares


def train_loop(tag, model, groups_, loss_fn, eval_fn, fit_idx, seed, epochs, init_best=None, first_batch_check=None):
    """Reference training loop (AdamW, cosine over `epochs`, clip, seeded shuffle, strictly-best val macro-F1)."""
    opt = torch.optim.AdamW(groups_, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=LR_MIN)
    params = [p for g in groups_ for p in g["params"]]
    loader = DataLoader(TensorDataset(torch.from_numpy(fit_idx)), batch_size=BATCH_SIZE, shuffle=True,
                        generator=make_generator(seed))
    best = init_best or {"f1": -1.0, "state": None, "epoch": None}
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        run, nb = 0.0, 0
        for (bi,) in loader:
            bi = bi.to(DEVICE)
            if first_batch_check is not None and ep == 1 and nb == 0:
                first_batch_check(bi)
            opt.zero_grad()
            loss = loss_fn(bi)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=GRAD_CLIP_NORM)
            opt.step()
            run += float(loss)
            nb += 1
        sched.step()
        assert math.isfinite(run), f"{tag} non-finite training loss"
        f1v = eval_fn()
        history.append((round(run / nb, 5), round(f1v, 5)))
        improved = f1v > best["f1"]
        if improved:
            best = {"f1": f1v, "state": copy.deepcopy(model.state_dict()), "epoch": ep}
        print(f"    {tag} epoch {ep:02d}/{epochs} | train loss {run / nb:.4f} | val macro-F1 {f1v:.4f}" + ("  * best" if improved else ""))
    model.load_state_dict(best["state"])
    return best, history


@torch.no_grad()
def predict(fn, idx, bs=256):
    """fn(batch_index_tensor) -> logits or (z_final, z_imu, alpha, dz). Returns probs (+ alpha, ||alpha dz|| if residual)."""
    probs, alphas, norms = [], [], []
    for i in range(0, len(idx), bs):
        bi = torch.from_numpy(idx[i:i + bs]).to(DEVICE)
        out = fn(bi)
        if isinstance(out, tuple):
            zf, _, a, dz = out
            assert bool(((a >= 0) & (a <= ALPHA_MAX)).all()), "alpha outside [0, ALPHA_MAX]"
            ALPHA_CHECKS.append(len(bi))
            alphas.append(a.squeeze(1).float().cpu().numpy())
            norms.append((a * dz).norm(dim=1).float().cpu().numpy())
            out = zf
        probs.append(torch.softmax(out, 1).float().cpu().numpy())
    P = np.concatenate(probs)
    return (P, np.concatenate(alphas), np.concatenate(norms)) if alphas else (P, None, None)


def val_f1(fn, idx):
    P, _, _ = predict(fn, idx)
    return f1_score(YW[idx], P.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0)


ALPHA_CHECKS, ZERO_INIT, ANCHOR_CHECKS, PERM_CHECKS, ID_HASHES = [], [], [], [], {}
C1_CHECKS = []                                     # E18b: C1 receives no audio (hook 10)


def run_arm(arm, k, seed, epochs=EPOCHS, r2_epochs=R2_EPOCHS):
    t0 = time.time()
    set_all_seeds(seed)
    f = FOLDS[k]
    P_ = FOLD_PREP[k]
    fit, va, te = P_["idx"]["train"], P_["idx"]["val"], P_["idx"]["test"]
    # ---- fail-loudly integrity checks ----
    assert f["day"] not in {MISSION_DAY[m] for m in f["train"] + f["val"]}, "test day in fitting/validation"
    assert {MISSION_CLASS[m] for m in f["val"]} == set(range(N_CLASSES)), "inner validation lacks a class"
    assert sha16(YW) == Y_HASH, "labels changed"
    id_hash = hashlib.sha1(("\n".join(WID[te]) + "|" + "\n".join(f["test"]) + "|" +
                            "\n".join(sorted(f["train"] + f["val"]))).encode()).hexdigest()[:16]
    assert ID_HASHES.setdefault((k, seed), id_hash) == id_hash, f"{arm}: test windows / missions / pool differ"
    XA_z, XS_z = zscored(k)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(P_["class_weights"], device=DEVICE))
    tag = f"[{arm} | fold {k} ({f['day']}) | seed {seed}]"
    print(f"{tag} fit {len(f['train'])} missions / {len(fit)} windows | val {len(f['val'])} / {len(va)} | test "
          f"{len(f['test'])} / {len(te)} | z {P_['zhash']}")
    extra = {"b1_w": np.nan, "zero_init_maxdiff": np.nan, "anchor_hash_before": "", "anchor_hash_after": "",
             "anchor_repro_maxdiff": np.nan, "c2_nonidentity_min": np.nan}
    alpha = norm = None
    best = {"epoch": None, "f1": np.nan}
    history = []
    init_sha1 = ""
    if arm in ("B0", "AUD"):
        model = (IMUMamba() if arm == "B0" else AudioMamba()).to(DEVICE)
        assert verify_model(model)[0]
        init_sha1 = state_hash(model)
        fwd = (lambda bi: model(XS_z[bi])) if arm == "B0" else (lambda bi: model(XA_z[bi]))
        best, history = train_loop(tag, model, [{"params": list(model.parameters())}], lambda bi: ce(fwd(bi), Y_G[bi]),
                                   lambda: val_f1(fwd, va), fit, seed, epochs)
        model.eval()
        Pt, _, _ = predict(fwd, te)
        Pv, _, _ = predict(fwd, va)
        torch.save(best["state"], CKPT_DIR / f"{arm}_{k}_{seed}.pt")
    elif arm == "B1":
        zb, za = np.load(PROBS_DIR / f"B0_{k}_{seed}.npz"), np.load(PROBS_DIR / f"AUD_{k}_{seed}.npz")
        assert np.array_equal(zb["val_window_id"], WID[va]) and np.array_equal(za["window_id"], WID[te])
        scores = {w: f1_score(YW[va], (w * zb["probs_val"] + (1 - w) * za["probs_val"]).argmax(1), average="macro",
                              labels=list(range(N_CLASSES)), zero_division=0) for w in B1_WEIGHTS}
        w_best = max(B1_WEIGHTS, key=lambda w: (round(scores[w], 12), w))     # ties -> larger w
        print(f"{tag} validation macro-F1 per w: " + ", ".join(f"{w}: {s:.4f}" for w, s in scores.items()) + f" -> w = {w_best}")
        Pt = w_best * zb["probs"] + (1 - w_best) * za["probs"]
        Pv = w_best * zb["probs_val"] + (1 - w_best) * za["probs_val"]
        extra["b1_w"] = w_best
        best = {"epoch": 0, "f1": scores[w_best]}
    elif arm == "B1c":                                  # E18b: E18 B1 code with C1 in place of B0
        zb, za = np.load(PROBS_DIR / f"C1_{k}_{seed}.npz"), np.load(PROBS_DIR / f"AUD_{k}_{seed}.npz")
        assert np.array_equal(zb["val_window_id"], WID[va]) and np.array_equal(za["window_id"], WID[te])
        scores = {w: f1_score(YW[va], (w * zb["probs_val"] + (1 - w) * za["probs_val"]).argmax(1), average="macro",
                              labels=list(range(N_CLASSES)), zero_division=0) for w in B1_WEIGHTS}
        w_best = max(B1_WEIGHTS, key=lambda w: (round(scores[w], 12), w))     # ties -> larger w
        print(f"{tag} validation macro-F1 per w: " + ", ".join(f"{w}: {s:.4f}" for w, s in scores.items()) + f" -> w = {w_best}")
        Pt = w_best * zb["probs"] + (1 - w_best) * za["probs"]
        Pv = w_best * zb["probs_val"] + (1 - w_best) * za["probs_val"]
        extra["b1_w"] = w_best
        best = {"epoch": 0, "f1": scores[w_best]}
    elif arm == "B2":
        model = JointFusion().to(DEVICE)
        init_sha1 = state_hash(model)
        fwd = lambda bi: model(XA_z[bi], XS_z[bi])
        best, history = train_loop(tag, model, [{"params": list(model.parameters())}], lambda bi: ce(fwd(bi), Y_G[bi]),
                                   lambda: val_f1(fwd, va), fit, seed, epochs)
        model.eval()
        Pt, _, _ = predict(fwd, te)
        Pv, _, _ = predict(fwd, va)
    else:                                                  # R1, C1, C2, R2: residual fusion around the B0 anchor
        anchor = IMUMamba().to(DEVICE)
        anchor.load_state_dict(torch.load(CKPT_DIR / f"B0_{k}_{seed}.pt", map_location=DEVICE))
        model = ResidualFusion(anchor, use_audio=(arm != "C1")).to(DEVICE)
        xa_src = XA_z
        if arm == "C2":
            perm, shares = c2_permutation(k, seed)
            xa_src = XA_z[torch.from_numpy(perm).to(DEVICE)]
            extra["c2_nonidentity_min"] = min(shares.values())
            PERM_CHECKS.append((k, seed, shares))
            print(f"{tag} C2 permutation non-identity share per split: {shares}")
        fwd = lambda bi: model(xa_src[bi], XS_z[bi])

        def lossR(bi):
            zf, zi, a, dz = fwd(bi)
            loss = ce(zf, Y_G[bi]) + LAMBDA_R * ((a * dz) ** 2).sum(dim=1).mean()
            if arm == "R2":
                loss = loss + LAMBDA_I * ce(zi, Y_G[bi])
            return loss

        if arm == "R2":
            model.load_state_dict(torch.load(CKPT_DIR / f"R1_{k}_{seed}.pt", map_location=DEVICE))
            model.set_anchor_frozen(False)
            init_sha1 = state_hash(model)
            r1_f1 = val_f1(fwd, va)
            print(f"{tag} incumbent = R1 best state, validation macro-F1 {r1_f1:.4f} (selected_epoch 0 if never beaten)")
            other = [p for n_, p in model.named_parameters() if not n_.startswith("anchor.")]
            best, history = train_loop(tag, model, [{"params": list(model.anchor.parameters()),
                                                     "lr": LEARNING_RATE * R2_IMU_LR_FACTOR},
                                                    {"params": other, "lr": LEARNING_RATE}], lossR,
                                       lambda: val_f1(fwd, va), fit, seed, r2_epochs,
                                       init_best={"f1": r1_f1, "state": copy.deepcopy(model.state_dict()), "epoch": 0})
        else:
            init_sha1 = state_hash(model)
            extra["anchor_hash_before"] = state_hash(model.anchor)
            Pb = np.load(PROBS_DIR / f"B0_{k}_{seed}.npz")["probs"]
            Pa, _, _ = predict(lambda bi: model.anchor(XS_z[bi])[1], te)
            extra["anchor_repro_maxdiff"] = float(np.abs(Pa - Pb).max())
            assert extra["anchor_repro_maxdiff"] < ANCHOR_REPRO_TOL, f"{tag}: anchor does not reproduce B0 ({extra['anchor_repro_maxdiff']:.2e})"

            def zero_check(bi):
                with torch.no_grad():
                    zf, zi, _, _ = fwd(bi)
                d = float((zf - zi).abs().max())
                extra["zero_init_maxdiff"] = d
                ZERO_INIT.append(d)
                assert d < ZERO_INIT_TOL, f"{tag}: z_final != z_IMU at step 0 (max |diff| {d:.2e})"

            trainable = [p for p in model.parameters() if p.requires_grad]
            assert not any(p.requires_grad for p in model.anchor.parameters())
            best, history = train_loop(tag, model, [{"params": trainable}], lossR, lambda: val_f1(fwd, va), fit, seed,
                                       epochs, first_batch_check=zero_check)
            extra["anchor_hash_after"] = state_hash(model.anchor)
            assert extra["anchor_hash_before"] == extra["anchor_hash_after"], f"{tag}: frozen anchor changed"
            ANCHOR_CHECKS.append((arm, k, seed, extra["anchor_repro_maxdiff"]))
        if arm == "C1":                                    # E18b: C1 must receive no audio
            assert model.aud is None, f"{tag}: C1 has an audio encoder"
            _bi = torch.from_numpy(te[:64]).to(DEVICE)
            model.eval()
            with torch.no_grad():
                _d = float((model(XA_z[_bi], XS_z[_bi])[0] - model(torch.zeros_like(XA_z[_bi]), XS_z[_bi])[0]).abs().max())
            assert _d == 0.0, f"{tag}: C1 output depends on audio (max |diff| {_d})"
            C1_CHECKS.append(True)
        model.eval()
        Pt, alpha, norm = predict(fwd, te)
        Pv, _, _ = predict(fwd, va)
        if arm == "R1":
            torch.save(best["state"], CKPT_DIR / f"R1_{k}_{seed}.pt")
    yt = YW[te]
    pred = Pt.argmax(1)
    my_, mP_, _ = mission_level(WMIS[te], yt, Pt)
    rec = np.array([((pred == c) & (yt == c)).sum() / max((yt == c).sum(), 1) for c in range(N_CLASSES)])
    res = {"arm": arm, "fold": k, "fold_day": f["day"], "seed": seed,
           "window_F1": f1_score(yt, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0),
           "mission_F1": f1_score(my_, mP_.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0),
           "window_accuracy": float((pred == yt).mean()), **{f"recall_c{c}": float(rec[c]) for c in range(N_CLASSES)},
           "faulty_called_normal": float((pred[yt > 0] == 0).mean()), "selected_epoch": best["epoch"],
           "best_val_macro_f1": round(float(best["f1"]), 6), "n_train_missions": len(f["train"]),
           "n_val_missions": len(f["val"]), "n_test_missions": len(f["test"]), "n_test_windows": len(te),
           "id_hash": id_hash, "zstats_sha1": P_["zhash"], "init_sha1": init_sha1, **extra,
           "mean_alpha": float(alpha.mean()) if alpha is not None else np.nan,
           "mean_corr_norm": float(norm.mean()) if norm is not None else np.nan,
           "epochs": r2_epochs if arm == "R2" else (0 if arm == "B1" else epochs), "inner_val_rule": INNER_VAL_RULE,
           "train_seconds": round(time.time() - t0, 1), "history": json.dumps(history)}
    np.savez(PROBS_DIR / f"{arm}_{k}_{seed}.npz", window_id=WID[te].astype(str), mission=WMIS[te].astype(str), cls=yt,
             day=WDAY[te].astype(str), phase=WPHASE[te].astype(str), probs=Pt.astype(np.float32),
             val_window_id=WID[va].astype(str), probs_val=Pv.astype(np.float32),
             alpha=alpha if alpha is not None else np.zeros(0, np.float32),
             corr_norm=norm if norm is not None else np.zeros(0, np.float32))
    if arm not in ("B1", "B1c"):
        del model
    gc.collect()
    torch.cuda.empty_cache()
    return res

## 5. Wavelet-scattering audio tokens (D2, D2sh)
- **Same coverage as R0.** Per window, the scattering uses the **same 64 chunks** that v29 picked for the R0 tokens.
  v29's `paired_windows` does not store which chunks it picked, so the picks are recomputed with v29's own rule
  (`a_idx[linspace(0, n − 1, 64).round()]`). For **every** window it is asserted that the recomputed chunks
  reproduce `XA` exactly.
- **Raw audio** comes from the AudioBuffer JSONL, read with v29's line filter. The chunks are kept in the same order
  as v29's `feats`, which is asserted by recomputing R0 from the raw samples.
- **Transform.** Per chunk and channel, the 256 samples (per-chunk mean removed, as in R0) go through a 1D wavelet
  scattering transform:
  - kymatio `Scattering1D(J = 6, Q = 2, max_order = 2, T = 2^J)`, averaged over time to one value per path;
  - if kymatio cannot be installed or fails its sanity test, an own Morlet filter-bank scattering in PyTorch is used
    instead. **The backend used is printed.**
- **Compression and z-score.** log(1 + |S| / 1e-3). The order-0 coefficient is signed, so |·| keeps it finite. The
  z-score uses fit windows only, per fold.
- **Caching.** Computed once per chunk and cached to disk (`e23_scatter_cache/`, float32). It uses no labels and no
  statistics, so it is fold-independent.
- **Frequencies** are **normalised (cycles per sample), not Hz** (the audio sample rate is unverified).

In [ ]:
# ==============================================================================
# 5a. SCATTERING BACKEND (kymatio Scattering1D, else an own Morlet filter-bank scattering) + SANITY (HOOK 15, part 1)
# ==============================================================================
SCAT_DIR = OUT / "e23_scatter_cache"
SCAT_DIR.mkdir(parents=True, exist_ok=True)
SCAT_BACKEND, _scat_batch, _kym = None, None, None


def _first_order_test(scat_fn, order, xi):
    """A sinusoid at a first-order path's centre frequency must activate that path most (among first-order paths)."""
    fo = np.where(order == 1)[0]
    ok = True
    for j in (fo[len(fo) // 4], fo[len(fo) // 2], fo[(3 * len(fo)) // 4]):
        n = torch.arange(FRAME, device=DEVICE, dtype=torch.float32)
        x = torch.sin(2 * math.pi * float(xi[j]) * n)[None]
        with torch.no_grad():
            s = scat_fn(x)[0].float().cpu().numpy()
        nearest = fo[np.argmin(np.abs(xi[fo] - xi[j]))]
        got = fo[np.argmax(s[fo])]
        print(f"  sinusoid at {xi[j]:.4f} cycles/sample -> strongest first-order path {got} (centre {xi[got]:.4f}); "
              f"nearest centre: path {nearest}")
        ok &= got == nearest
    return bool(ok)


try:
    if importlib.util.find_spec("kymatio") is None:
        _pip(["kymatio"])
        importlib.invalidate_caches()
    import kymatio
    from kymatio.torch import Scattering1D
    _kym = Scattering1D(J=SCAT_J, shape=FRAME, Q=SCAT_Q, max_order=SCAT_MAX_ORDER, T=2 ** SCAT_J).to(DEVICE)
    _meta = _kym.meta()
    SCAT_ORDER = np.asarray(_meta["order"]).astype(int).ravel()
    _xi = np.asarray(_meta["xi"], dtype=np.float64)
    _xi = _xi[:, None] if _xi.ndim == 1 else _xi
    SCAT_XI = np.array([0.0 if np.all(np.isnan(r)) else float(r[~np.isnan(r)][-1]) for r in _xi])
    with torch.no_grad():
        _probe = _kym(torch.zeros(2, FRAME, device=DEVICE))
    assert _probe.shape[-2] == len(SCAT_ORDER), f"kymatio output paths {_probe.shape} vs meta {len(SCAT_ORDER)}"
    _scat_fn = lambda x: _kym(x).mean(dim=-1)                                  # (B, P): averaged over time
    if not _first_order_test(_scat_fn, SCAT_ORDER, SCAT_XI):
        raise RuntimeError("kymatio failed the sinusoid sanity test")
    SCAT_BACKEND = (f"kymatio {kymatio.__version__} Scattering1D(J={SCAT_J}, Q={SCAT_Q}, max_order={SCAT_MAX_ORDER}, "
                    f"T=2^{SCAT_J}), averaged over time")
    _scat_batch = SCAT_BATCH_KYMATIO
except Exception as exc:
    loud(f"kymatio Scattering1D unavailable or failed ({exc!r}).\nUsing the OWN Morlet filter-bank scattering (PyTorch).")
    _kym = None

if SCAT_BACKEND is None:
    _w = torch.fft.fftfreq(FRAME, device=DEVICE).double()

    def _bank(q, n):
        xis = [MORLET_XI_MAX * 2 ** (-l / q) for l in range(n)]
        r = 2 ** (-1 / q)
        filt = [torch.where(_w > 0, torch.exp(-(_w - x) ** 2 / (2 * (x * (1 - r) / 2.355) ** 2)), torch.zeros_like(_w))
                for x in xis]                                        # analytic Gaussian (Morlet-like) bands, peak 1
        return np.array(xis), torch.stack(filt).to(torch.complex64)

    _XI1, _PSI1 = _bank(SCAT_Q, SCAT_J * SCAT_Q)                     # first order: J*Q filters
    _XI2, _PSI2 = _bank(1, SCAT_J)                                   # second order: J filters (Q2 = 1)
    _PAIRS = [(i1, i2) for i1 in range(len(_XI1)) for i2 in range(len(_XI2)) if _XI2[i2] < _XI1[i1]]

    def _scat_fn(x):
        X = torch.fft.fft(x.to(torch.complex64), dim=-1)
        U1 = torch.fft.ifft(X[:, None, :] * _PSI1[None], dim=-1).abs()            # (B, J*Q, N)
        S0 = x.mean(dim=-1, keepdim=True)
        S1 = U1.mean(dim=-1)
        U1f = torch.fft.fft(U1.to(torch.complex64), dim=-1)
        S2 = torch.stack([torch.fft.ifft(U1f[:, i1] * _PSI2[i2], dim=-1).abs().mean(dim=-1) for i1, i2 in _PAIRS], dim=1)
        return torch.cat([S0, S1, S2], dim=1)

    SCAT_ORDER = np.array([0] + [1] * len(_XI1) + [2] * len(_PAIRS))
    SCAT_XI = np.concatenate([[0.0], _XI1, [_XI2[i2] for _, i2 in _PAIRS]])
    _ok_own = _first_order_test(_scat_fn, SCAT_ORDER, SCAT_XI)
    hook(15, _ok_own, "own Morlet scattering: a sinusoid activates the nearest first-order path", stop=True)
    SCAT_BACKEND = (f"OWN Morlet filter-bank scattering (PyTorch; J={SCAT_J}, Q={SCAT_Q}, second order Q=1, "
                    f"averaged over the 256 samples)")
    _scat_batch = SCAT_BATCH_OWN
N_SCAT_PATHS = len(SCAT_ORDER)
print(f"SCATTERING BACKEND: {SCAT_BACKEND} | paths {N_SCAT_PATHS} (order 0: {int((SCAT_ORDER == 0).sum())}, order 1: "
      f"{int((SCAT_ORDER == 1).sum())}, order 2: {int((SCAT_ORDER == 2).sum())}) | token feature size "
      f"{N_CH} channels x {N_SCAT_PATHS} paths = {N_CH * N_SCAT_PATHS} | frequencies are NORMALISED (cycles/sample)")
hook(15, True, f"backend {SCAT_BACKEND}: sinusoid test passed (asserted above)", stop=True)

In [ ]:
# ==============================================================================
# 5b. SCATTERING PER CHUNK (cached per mission) -> window tokens on v29's chunk picks (HOOK 15, part 2)
# ==============================================================================
NAME2MDIR = {d.name: d for d in ALL_DIRS}


def raw_chunks(name):
    """Raw AudioBuffer chunks of a mission with v29's line filter and modal-length rule, in v29's `feats` order:
    returns (n_kept, 5 channels, 256) float32 with the per-chunk channel mean removed (as for R0)."""
    a_path = sorted(NAME2MDIR[name].glob("*AudioBuffer*.jsonl"))[0]
    chunks, lens = [], Counter()
    with open(a_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            x = rec.get("data")
            if x is None or len(x) == 0:
                continue
            lens[len(x)] += 1
            chunks.append(x)
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{name}: modal buffer length {modal}"
    raw = np.asarray([x for x in chunks if len(x) == modal], dtype=np.int64)
    del chunks
    ch = raw.reshape(len(raw), FRAME, N_SLOTS)[:, :, KEEP_SLOTS].astype(np.float64)    # (n, 256, 5)
    ch = ch - ch.mean(axis=1, keepdims=True)
    return ch


_t0 = time.time()
_S_BY_MISSION = {}
for i_, name in enumerate(KEPT, 1):
    cache = SCAT_DIR / f"{name}.npz"
    if cache.exists():
        _S_BY_MISSION[name] = np.load(cache)["S"]
    else:
        ch = raw_chunks(name)
        pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1)
        r0 = (pw[:, :, KEEP_BINS] ** 0.25).reshape(len(pw), N_AUDIO_FEAT)
        assert r0.shape == PARSED[name]["feats"].shape and np.allclose(r0, PARSED[name]["feats"], rtol=1e-4, atol=1e-3), \
            f"{name}: raw chunks do not reproduce v29's R0 features (chunk order differs)"
        seg = torch.from_numpy(ch.transpose(0, 2, 1).reshape(-1, FRAME).astype(np.float32))           # (n*5, 256)
        out = []
        with torch.no_grad():
            for b0 in range(0, len(seg), _scat_batch):
                out.append(_scat_fn(seg[b0:b0 + _scat_batch].to(DEVICE)).float().cpu())
        S = torch.cat(out).numpy().reshape(len(ch), N_CH, N_SCAT_PATHS).astype(np.float32)
        S[:, :, SCAT_ORDER == 0] = np.abs(S[:, :, SCAT_ORDER == 0])         # signed order-0 -> |.| (finite log)
        assert np.isfinite(S).all(), f"{name}: non-finite scattering coefficients"
        np.savez(SCAT_DIR / f"{name}.tmp.npz", S=S)
        os.replace(SCAT_DIR / f"{name}.tmp.npz", cache)
        _S_BY_MISSION[name] = S
        del ch, pw, r0, seg, out
    if i_ % 10 == 0 or i_ == len(KEPT):
        print(f"  scattering {i_}/{len(KEPT)} missions | {time.time() - _t0:.0f} s", flush=True)

# ---- window tokens on v29's chunk picks (recomputed with v29's rule and checked against XA for EVERY window) ----
SCW = np.empty((len(WMETA), TA_AUDIO, N_CH, N_SCAT_PATHS), np.float32)
_aligned = 0
for name in KEPT:
    _, a_times, _, _, _ = timebase(PARSED[name])
    rows = np.where(WMETA.mission.values == name)[0]
    for r_ in rows:
        a_mask = (a_times >= WMETA.t_start.values[r_]) & (a_times <= WMETA.t_end.values[r_])
        assert a_mask.sum() >= 2, f"{WMETA.window_id.values[r_]}: fewer than 2 real chunks"
        a_idx = np.where(a_mask)[0]
        a_pick = a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]
        assert np.array_equal(PARSED[name]["feats"][a_pick], XA[r_]), f"{WMETA.window_id.values[r_]}: chunk picks differ"
        SCW[r_] = _S_BY_MISSION[name][a_pick]
        _aligned += 1
del _S_BY_MISSION
SCAT_WIN_ENERGY = (SCW.astype(np.float64) ** 2).mean(axis=(1, 2))                 # (N, P): for the strong/weak split
SCL_G = torch.from_numpy(np.log1p(SCW / SCAT_EPS).reshape(len(WMETA), TA_AUDIO, N_CH * N_SCAT_PATHS)).to(DEVICE)
_fin = bool(torch.isfinite(SCL_G).all())
hook(15, _fin and _aligned == len(WMETA) and tuple(SCL_G.shape) == (len(WMETA), TA_AUDIO, N_CH * N_SCAT_PATHS),
     f"scattering tokens {tuple(SCL_G.shape)} finite={_fin}; every one of {_aligned} windows uses exactly v29's 64 chunks "
     f"(features reproduce XA); raw chunks reproduce v29's R0 features", stop=True)
del SCW
gc.collect()
torch.cuda.empty_cache()
mem_report("after scattering tokens")

## 6. Strong / weak frequency split (FIT windows of each fold only) and band-limited IMU streams
- **Audio, D1:** the 16 FFT bins are ranked by mean energy (feature⁴ = power) over fit windows, tokens and channels.
  The top half is STRONG and the bottom half WEAK, with all 5 channels of a bin in the same set.
  - **D1r** assigns the same number of bins to each set at random (seed [2033, fold]).
- **Audio, D2:** the scattering paths are ranked by mean energy (S²) over fit windows, tokens and channels, and split
  into halves the same way.
- **IMU:** the rFFT (65 bins) of each 128-sample window per channel; the bins are ranked by fit-window mean energy and
  split into halves.
  - Two band-limited IMU streams (128 × 6 each) are built by masking the spectrum and inverting.
  - Each stream is z-scored per channel on fit windows.
  - This is precomputed once per fold (numpy).
- **Hook 14** asserts that the sets come from fit windows only, are disjoint and complete, and that D1r has D1's
  sizes. The strong set per fold and the Jaccard overlap across folds are printed and saved to `e23_split_sets.csv`.

In [ ]:
# ==============================================================================
# 6a. STRONG / WEAK SETS PER FOLD (fit windows only) + band-limited IMU streams + scattering z-score (HOOK 14)
# ==============================================================================
N_FFT_BINS = len(KEEP_BINS)
AUD_WIN_ENERGY = (XA.astype(np.float64) ** 4).reshape(len(XA), TA_AUDIO, N_CH, N_FFT_BINS).mean(axis=(1, 2))   # (N, 16)
SPEC_IMU = np.fft.rfft(XS.astype(np.float64), axis=1).astype(np.complex64)                       # (N, 65, 6)
IMU_WIN_ENERGY = (np.abs(SPEC_IMU).astype(np.float64) ** 2).mean(axis=2)                          # (N, 65)
XS_HASH = sha16(XS)


def top_half(energy):
    order = np.argsort(-np.asarray(energy), kind="stable")
    n_s = len(order) // 2
    return sorted(int(i) for i in order[:n_s]), sorted(int(i) for i in order[n_s:])


SPLITS, _rows, _ok14 = {}, [], True
for k in FOLDS:
    fit = FOLD_PREP[k]["idx"]["train"]
    _ok14 &= set(WMIS[fit]) == set(FOLDS[k]["train"])                                            # fit windows only
    SPLITS[("audio_fft", k)] = top_half(AUD_WIN_ENERGY[fit].mean(axis=0))
    _perm = np.random.default_rng([D1R_SEED, k]).permutation(N_FFT_BINS)
    _ns = len(SPLITS[("audio_fft", k)][0])
    SPLITS[("audio_fft_random", k)] = (sorted(int(i) for i in _perm[:_ns]), sorted(int(i) for i in _perm[_ns:]))
    SPLITS[("audio_scat", k)] = top_half(SCAT_WIN_ENERGY[fit].mean(axis=0))
    SPLITS[("imu", k)] = top_half(IMU_WIN_ENERGY[fit].mean(axis=0))
    for kind, n_all in (("audio_fft", N_FFT_BINS), ("audio_fft_random", N_FFT_BINS), ("audio_scat", N_SCAT_PATHS),
                        ("imu", SPEC_IMU.shape[1])):
        s_, w_ = SPLITS[(kind, k)]
        _ok14 &= not (set(s_) & set(w_)) and set(s_) | set(w_) == set(range(n_all))
        _rows.append({"fold": k, "test_day": FOLDS[k]["day"], "kind": kind, "n_strong": len(s_), "n_weak": len(w_),
                      "strong": json.dumps(s_), "weak": json.dumps(w_)})
    _ok14 &= len(SPLITS[("audio_fft_random", k)][0]) == len(SPLITS[("audio_fft", k)][0])
SPLIT_TABLE = pd.DataFrame(_rows)
SPLIT_TABLE.to_csv(OUT / "e23_split_sets.csv", index=False)
print("STRONG sets per fold (audio FFT: bin index 0..15 = FFT bin 1..16; scattering: path index; IMU: rFFT bin 0..64):")
for kind in ("audio_fft", "audio_fft_random", "audio_scat", "imu"):
    for k in FOLDS:
        print(f"  {kind:<17} fold {k}: strong {SPLITS[(kind, k)][0]}")
    _j = [len(set(SPLITS[(kind, a)][0]) & set(SPLITS[(kind, b)][0])) / len(set(SPLITS[(kind, a)][0]) | set(SPLITS[(kind, b)][0]))
          for a in FOLDS for b in FOLDS if a < b]
    print(f"  {kind:<17} Jaccard overlap of strong sets across folds: mean {np.mean(_j):.3f}, min {np.min(_j):.3f}")
hook(14, _ok14, "strong/weak sets from fit windows only; disjoint and complete for audio FFT, D1r, scattering and IMU; "
                "D1r sizes equal to D1's -> e23_split_sets.csv", stop=True)


def audio_cols(kind, k):
    """Column indices of the strong and weak audio feature sets (all channels of a selected bin / path)."""
    s_, w_ = SPLITS[(kind, k)]
    width = N_FFT_BINS if kind.startswith("audio_fft") else N_SCAT_PATHS
    cols = lambda sel: torch.tensor([c_ * width + j for c_ in range(N_CH) for j in sel], device=DEVICE, dtype=torch.long)
    return cols(s_), cols(w_)


_IMU_STREAMS = {}


def imu_streams_z(k):
    """Band-limited IMU streams (strong, weak) for fold k: spectrum masked and inverted, z-scored on fit windows."""
    if k not in _IMU_STREAMS:
        fit = FOLD_PREP[k]["idx"]["train"]
        out = []
        for sel in SPLITS[("imu", k)]:
            mask = np.zeros(SPEC_IMU.shape[1], np.float32)
            mask[sel] = 1.0
            x = np.fft.irfft(SPEC_IMU * mask[None, :, None], n=TS_SENSOR, axis=1).astype(np.float32)   # (N, 128, 6)
            mu = x[fit].reshape(-1, x.shape[2]).mean(0, dtype=np.float64)
            sd = x[fit].reshape(-1, x.shape[2]).std(0, dtype=np.float64) + SENSOR_STD_EPS
            out.append(((x - mu) / sd).astype(np.float32))
        _IMU_STREAMS[k] = out
    return tuple(torch.from_numpy(a_).to(DEVICE) for a_ in _IMU_STREAMS[k])


def scat_z(k):
    """Scattering tokens z-scored with the fold's FIT-window statistics (GPU; freed by the caller)."""
    tr = torch.from_numpy(FOLD_PREP[k]["idx"]["train"]).to(DEVICE)
    flat = SCL_G[tr].reshape(-1, SCL_G.shape[-1]).double()
    mu, sd = flat.mean(0).float(), (flat.std(0, unbiased=False) + AUDIO_STD_EPS).float()
    del flat
    return (SCL_G - mu) / sd

## 7. Dual-path module (the same code for D1, D1r, D2, D1sh, D2sh)
- **Tokenisers** (trainable), each with a learned positional embedding. No HTT, no LACA, no timestamp bias.
  - Audio strong / weak: `Linear(F_p → 64)` over the 64 time tokens.
  - IMU strong / weak: `Conv1d(6 → 64, k = 4, stride 2, padding 1)`, which turns 128 samples into 64 tokens.
- **Per path p ∈ {strong, weak}:** one **bidirectional** 4-head cross-attention layer, each direction with a residual
  connection and LayerNorm.
  - audio_p attends over IMU_p;
  - IMU_p attends over audio_p.
- **Pooling and concatenation:** each of the 4 outputs is pooled with **mean ‖ attention pooling**. The four pooled
  vectors are concatenated with the frozen anchor's IMU features h_I.
- **Correction head:** identical to R1 — hidden 128, the last layer zero-initialised, α = ALPHA_MAX · sigmoid(g(·))
  on the same concatenated input, z_final = z_IMU + α·Δz, loss CE + LAMBDA_R · mean‖αΔz‖². At step 0
  z_final == z_IMU (asserted, < 1e-5).
- **Path ablation** (evaluation only): `model.ablate = "strong"` or `"weak"` zeros that path's pooled outputs.

In [ ]:
# ==============================================================================
# 7a. DUAL-PATH FUSION + run_dual (reuses E18b's train_loop / predict / val_f1 / Pooled / c2_permutation) + wrapper
# ==============================================================================
class DualPathFusion(nn.Module):
    """Frozen IMU anchor + strong/weak bidirectional cross-attention paths + R1's zero-initialised correction head."""

    def __init__(self, anchor, fa_strong, fa_weak):
        super().__init__()
        d = DUAL_D_MODEL
        self.anchor = Pooled(anchor)
        self.tok_a = nn.ModuleDict({"strong": nn.Linear(fa_strong, d), "weak": nn.Linear(fa_weak, d)})
        self.tok_i = nn.ModuleDict({p: nn.Conv1d(N_SENSOR_CHANNELS, d, kernel_size=4, stride=2, padding=1) for p in PATHS})
        self.pos = nn.ParameterDict({f"{m}_{p}": nn.Parameter(DUAL_POS_INIT_STD * torch.randn(1, TA_AUDIO, d))
                                     for m in ("a", "i") for p in PATHS})
        self.att_ai = nn.ModuleDict({p: nn.MultiheadAttention(d, DUAL_HEADS, batch_first=True) for p in PATHS})
        self.att_ia = nn.ModuleDict({p: nn.MultiheadAttention(d, DUAL_HEADS, batch_first=True) for p in PATHS})
        self.ln_a = nn.ModuleDict({p: nn.LayerNorm(d) for p in PATHS})
        self.ln_i = nn.ModuleDict({p: nn.LayerNorm(d) for p in PATHS})
        self.pool = nn.ModuleDict({f"{m}_{p}": nn.Linear(d, 1) for m in ("a", "i") for p in PATHS})
        d_in = D_MODEL + 4 * 2 * d                                   # h_I + 4 streams x (mean || attention pool)
        self.w1 = nn.Linear(d_in, R_HIDDEN)
        self.w2 = nn.Linear(R_HIDDEN, N_CLASSES)
        nn.init.zeros_(self.w2.weight)
        nn.init.zeros_(self.w2.bias)
        self.gate = nn.Linear(d_in, 1)
        self.ablate = None
        self.anchor_frozen = True
        for p_ in self.anchor.parameters():
            p_.requires_grad_(False)
        self.anchor.eval()

    def train(self, mode=True):
        super().train(mode)
        self.anchor.eval()                                           # the frozen anchor never updates BatchNorm
        return self

    def _pool(self, x, key):
        w = F.softmax(self.pool[key](x), dim=1)
        return torch.cat([x.mean(dim=1), (x * w).sum(dim=1)], dim=1)

    def forward(self, xa_s, xa_w, xi_s, xi_w, xs):
        with torch.no_grad():
            hi, zi = self.anchor(xs)
        feats = [hi]
        for p, xa, xi in (("strong", xa_s, xi_s), ("weak", xa_w, xi_w)):
            a = self.tok_a[p](xa) + self.pos[f"a_{p}"]
            i = self.tok_i[p](xi.transpose(1, 2)).transpose(1, 2) + self.pos[f"i_{p}"]
            a2 = self.ln_a[p](a + self.att_ai[p](a, i, i, need_weights=False)[0])
            i2 = self.ln_i[p](i + self.att_ia[p](i, a, a, need_weights=False)[0])
            pa, pi = self._pool(a2, f"a_{p}"), self._pool(i2, f"i_{p}")
            if self.ablate == p:
                pa, pi = torch.zeros_like(pa), torch.zeros_like(pi)
            feats += [pa, pi]
        h = torch.cat(feats, dim=1)
        dz = self.w2(F.relu(self.w1(h)))
        alpha = ALPHA_MAX * torch.sigmoid(self.gate(h))
        return zi + alpha * dz, zi, alpha, dz


def run_dual(arm, k, seed, epochs=EPOCHS):
    t0 = time.time()
    set_all_seeds(seed)
    f = FOLDS[k]
    P_ = FOLD_PREP[k]
    fit, va, te = P_["idx"]["train"], P_["idx"]["val"], P_["idx"]["test"]
    # ---- the same fail-loudly checks as E18b's run_arm ----
    assert f["day"] not in {MISSION_DAY[m] for m in f["train"] + f["val"]}, "test day in fitting/validation"
    assert {MISSION_CLASS[m] for m in f["val"]} == set(range(N_CLASSES)), "inner validation lacks a class"
    assert sha16(YW) == Y_HASH, "labels changed"
    id_hash = hashlib.sha1(("\n".join(WID[te]) + "|" + "\n".join(f["test"]) + "|" +
                            "\n".join(sorted(f["train"] + f["val"]))).encode()).hexdigest()[:16]
    assert ID_HASHES.setdefault((k, seed), id_hash) == id_hash, f"{arm}: test windows / missions / pool differ"
    tag = f"[{arm} | fold {k} ({f['day']}) | seed {seed}]"
    kind = {"D1": "audio_fft", "D1sh": "audio_fft", "D1r": "audio_fft_random", "D2": "audio_scat", "D2sh": "audio_scat"}[arm]
    XA_z, XS_z = zscored(k)                                           # E18b's train-only z-score (R0 audio, IMU)
    if kind == "audio_scat":
        del XA_z
        XA_z = scat_z(k)
    cs, cw = audio_cols(kind, k)
    extra = {"b1_w": np.nan, "zero_init_maxdiff": np.nan, "anchor_hash_before": "", "anchor_hash_after": "",
             "anchor_repro_maxdiff": np.nan, "c2_nonidentity_min": np.nan,
             "split_hash": hashlib.sha1(json.dumps([SPLITS[(kind, k)], SPLITS[("imu", k)]]).encode()).hexdigest()[:12]}
    if arm.endswith("sh"):
        perm, shares = c2_permutation(k, seed)                        # E18b's C2 rule (C2_PERM_SEED, within split)
        XA_z = XA_z[torch.from_numpy(perm).to(DEVICE)]
        extra["c2_nonidentity_min"] = min(shares.values())
        PERM_CHECKS.append((k, seed, shares))
        print(f"{tag} audio shuffled within splits; non-identity share per split: {shares}")
    xi_s, xi_w = imu_streams_z(k)
    print(f"{tag} fit {len(f['train'])} missions / {len(fit)} windows | val {len(f['val'])} / {len(va)} | test "
          f"{len(f['test'])} / {len(te)} | audio {kind}: strong {len(cs)} / weak {len(cw)} columns | IMU bins strong "
          f"{len(SPLITS[('imu', k)][0])} / weak {len(SPLITS[('imu', k)][1])}")
    anchor = IMUMamba().to(DEVICE)
    anchor.load_state_dict(torch.load(CKPT_DIR / f"B0_{k}_{seed}.pt", map_location=DEVICE))
    model = DualPathFusion(anchor, len(cs), len(cw)).to(DEVICE)
    fwd = lambda bi: model(XA_z[bi][:, :, cs], XA_z[bi][:, :, cw], xi_s[bi], xi_w[bi], XS_z[bi])
    ce = nn.CrossEntropyLoss(weight=torch.tensor(P_["class_weights"], device=DEVICE))

    def lossR(bi):
        zf, zi, a, dz = fwd(bi)
        return ce(zf, Y_G[bi]) + LAMBDA_R * ((a * dz) ** 2).sum(dim=1).mean()

    init_sha1 = state_hash(model)
    extra["anchor_hash_before"] = state_hash(model.anchor)
    Pb = np.load(PROBS_DIR / f"B0_{k}_{seed}.npz")["probs"]
    Pa, _, _ = predict(lambda bi: model.anchor(XS_z[bi])[1], te)
    extra["anchor_repro_maxdiff"] = float(np.abs(Pa - Pb).max())
    assert extra["anchor_repro_maxdiff"] < ANCHOR_REPRO_TOL, f"{tag}: anchor does not reproduce B0"

    def zero_check(bi):
        with torch.no_grad():
            zf, zi, _, _ = fwd(bi)
        d_ = float((zf - zi).abs().max())
        extra["zero_init_maxdiff"] = d_
        ZERO_INIT.append(d_)
        assert d_ < ZERO_INIT_TOL, f"{tag}: z_final != z_IMU at step 0 (max |diff| {d_:.2e})"

    trainable = [p_ for p_ in model.parameters() if p_.requires_grad]
    assert not any(p_.requires_grad for p_ in model.anchor.parameters())
    best, history = train_loop(tag, model, [{"params": trainable}], lossR, lambda: val_f1(fwd, va), fit, seed, epochs,
                               first_batch_check=zero_check)
    extra["anchor_hash_after"] = state_hash(model.anchor)
    assert extra["anchor_hash_before"] == extra["anchor_hash_after"], f"{tag}: frozen anchor changed"
    ANCHOR_CHECKS.append((arm, k, seed, extra["anchor_repro_maxdiff"]))
    model.eval()
    Pt, alpha, norm = predict(fwd, te)
    Pv, _, _ = predict(fwd, va)
    abl = {}
    for p in PATHS:                                                   # eval-time path ablation (diagnostic)
        model.ablate = p
        abl[p], _, _ = predict(fwd, te)
    model.ablate = None
    assert sha16(XS) == XS_HASH and sha16(YW) == Y_HASH, "IMU or labels changed"
    yt = YW[te]
    pred = Pt.argmax(1)
    my_, mP_, _ = mission_level(WMIS[te], yt, Pt)
    rec = np.array([((pred == c_) & (yt == c_)).sum() / max((yt == c_).sum(), 1) for c_ in range(N_CLASSES)])
    _f1 = lambda P: f1_score(yt, P.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0)
    res = {"arm": arm, "fold": k, "fold_day": f["day"], "seed": seed, "window_F1": _f1(Pt),
           "mission_F1": f1_score(my_, mP_.argmax(1), average="macro", labels=list(range(N_CLASSES)), zero_division=0),
           "window_accuracy": float((pred == yt).mean()), **{f"recall_c{c_}": float(rec[c_]) for c_ in range(N_CLASSES)},
           "faulty_called_normal": float((pred[yt > 0] == 0).mean()), "selected_epoch": best["epoch"],
           "best_val_macro_f1": round(float(best["f1"]), 6), "n_train_missions": len(f["train"]),
           "n_val_missions": len(f["val"]), "n_test_missions": len(f["test"]), "n_test_windows": len(te),
           "id_hash": id_hash, "zstats_sha1": P_["zhash"], "init_sha1": init_sha1, **extra,
           "abl_strong_window_F1": _f1(abl["strong"]), "abl_weak_window_F1": _f1(abl["weak"]),
           "mean_alpha": float(alpha.mean()), "mean_corr_norm": float(norm.mean()), "epochs": epochs,
           "inner_val_rule": INNER_VAL_RULE, "train_seconds": round(time.time() - t0, 1), "history": json.dumps(history)}
    np.savez(PROBS_DIR / f"{arm}_{k}_{seed}.npz", window_id=WID[te].astype(str), mission=WMIS[te].astype(str), cls=yt,
             day=WDAY[te].astype(str), phase=WPHASE[te].astype(str), probs=Pt.astype(np.float32),
             val_window_id=WID[va].astype(str), probs_val=Pv.astype(np.float32), alpha=alpha, corr_norm=norm,
             probs_abl_strong=abl["strong"].astype(np.float32), probs_abl_weak=abl["weak"].astype(np.float32))
    model = anchor = XA_z = XS_z = xi_s = xi_w = None              # release GPU memory (the wrapper empties the cache)
    return res


def count_params(model):
    tot = sum(p_.numel() for p_ in model.parameters())
    trn = sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)
    return tot, trn


# ---- parameter count of every arm (built once with a dummy anchor; nothing is trained here) ----
_ns_fft = len(SPLITS[("audio_fft", 0)][0]) * N_CH, len(SPLITS[("audio_fft", 0)][1]) * N_CH
_ns_sct = len(SPLITS[("audio_scat", 0)][0]) * N_CH, len(SPLITS[("audio_scat", 0)][1]) * N_CH
_builders = {"B0": lambda: IMUMamba(), "C1": lambda: ResidualFusion(IMUMamba(), use_audio=False),
             "R1": lambda: ResidualFusion(IMUMamba(), use_audio=True),
             **{a: (lambda: DualPathFusion(IMUMamba(), *_ns_fft)) for a in ("D1", "D1r", "D1sh")},
             **{a: (lambda: DualPathFusion(IMUMamba(), *_ns_sct)) for a in ("D2", "D2sh")}}
_prow = []
for a in ARMS:
    _m = _builders[a]().to(DEVICE)
    tot, trn = count_params(_m)
    _prow.append({"arm": a, "total_params": tot, "trainable_params": trn if a != "B0" else tot,
                  "frozen_anchor_params": tot - trn if a != "B0" else 0})
    del _m
PARAMS = pd.DataFrame(_prow).set_index("arm")
PARAMS.to_csv(OUT / "e23_params.csv")
torch.cuda.empty_cache()
print("PARAMETER COUNTS (trainable = what the arm learns; residual arms also carry the frozen B0 anchor):")
print(PARAMS.to_string())


def run_arm_e23(arm, k, seed, epochs=EPOCHS, r2_epochs=R2_EPOCHS):
    """B0 / C1 / R1 -> E18b's run_arm (unchanged); dual-path arms -> run_dual. Frees memory after every run."""
    try:
        res = run_arm(arm, k, seed, epochs=epochs, r2_epochs=r2_epochs) if arm in ("B0", "C1", "R1") else \
            run_dual(arm, k, seed, epochs=epochs)
        res["n_trainable_params"] = int(PARAMS.loc[arm, "trainable_params"])
        print(f"  [{arm}] parameters: trainable {PARAMS.loc[arm, 'trainable_params']:,} | total {PARAMS.loc[arm, 'total_params']:,}")
        return res
    finally:
        gc.collect()
        torch.cuda.empty_cache()

## 8. Pre-registered decision rule (fixed BEFORE the run cell; also written to `e23_rule.txt`)

```text
PRE-REGISTERED DECISION RULE (E23; fixed before any E23 result; applied in code; never edited after results)
Primary metric: pooled out-of-fold WINDOW macro-F1 from seed-averaged probabilities on identical windows (the full v29
paired window set, 95 missions, day-out, seeds 11/21/31/41/51 — comparable with E18b).
PRIMARY CONTRASTS (K = 2): D1 vs R1 and D2 vs R1. A candidate is CREDITED only if ALL hold:
  1. pooled paired delta vs R1 >= +0.01 (the audio-specific part of R1 is only ~+0.02; each contrast is also reported
     against the stricter +0.02 line, descriptively);
  2. the 99% paired mission-cluster bootstrap CI (2000 resamples of missions within class, seed 2028, the SAME
     resampled missions for both arms) excludes 0;
  3. delta > 0 in >= 4 of 5 folds (seed-averaged, per held-out day);
  4. delta > 0 in >= 4 of 5 seeds (each seed's own pooled out-of-fold F1);
  5. worst-day F1 not below R1's;
  6. NO-HARM: Class-4 recall not more than 0.05 below R1's, and "faulty called Normal" not more than 0.02 above R1's;
  7. AUDIO ATTRIBUTION: the candidate beats its OWN shuffled control (D1 vs D1sh, D2 vs D2sh) by >= +0.02 with the
     99% CI excluding 0, >= 4/5 folds and >= 4/5 seeds.
WORSE (mirror): delta <= -0.01, the 99% CI entirely below 0, and delta < 0 in >= 4 of 5 folds. Otherwise NOT CREDITED.
SECONDARY (reported, conditions 1-5, no multiplicity claim): D2 vs D1 (scattering vs FFT); D1 vs D1r (energy split vs
random split); R1 vs C1, D1 vs C1, D2 vs C1, D1sh vs C1, D2sh vs C1 (capacity of the new head by itself; does R1
still replicate +0.02 over C1).
Also reported: per-day F1 for all arms, per-class recall, residual diagnostics (corrected / broken / net, mean alpha),
and the eval-time PATH ABLATION for D1 and D2 (zero the strong-path output, then the weak-path output; F1 change) as a
diagnostic of which path carries the signal. Every arm is reported whatever the outcome; a null result is valid.
No deployment assumptions: audio is always the plain N0 version.
```

In [ ]:
# ==============================================================================
# 5b. RUNS — seed-major, then fold, then the arm order B0, C1, R1, D1, D1r, D2, D1sh, D2sh (dependencies first).
#     CSV append after EVERY run; resumable on (arm, fold, seed); runtime estimate BEFORE training.
# ==============================================================================
CSV_FIELDS = ["arm", "fold", "fold_day", "seed", "window_F1", "mission_F1", "window_accuracy"] + \
             [f"recall_c{c}" for c in range(N_CLASSES)] + \
             ["faulty_called_normal", "selected_epoch", "best_val_macro_f1", "b1_w", "mean_alpha", "mean_corr_norm",
              "n_train_missions", "n_val_missions", "n_test_missions", "n_test_windows", "id_hash", "zstats_sha1",
              "init_sha1", "zero_init_maxdiff", "anchor_hash_before", "anchor_hash_after", "anchor_repro_maxdiff",
              "c2_nonidentity_min", "abl_strong_window_F1", "abl_weak_window_F1", "n_trainable_params", "split_hash", "epochs", "inner_val_rule", "train_seconds", "history", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    out = {}
    for r in df.itertuples():
        key = (r.arm, int(r.fold), int(r.seed))
        ok = (PROBS_DIR / f"{r.arm}_{int(r.fold)}_{int(r.seed)}.npz").exists()
        if r.arm in CKPT_ARMS:
            ok &= (CKPT_DIR / f"{r.arm}_{int(r.fold)}_{int(r.seed)}.pt").exists()
        if ok:
            out[key] = r
    return out


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


PLAN = [(a, k, s) for s in SEEDS_TO_RUN for k in FOLDS_TO_RUN for a in ARMS_TO_RUN]
_done = load_done()
_per_seed = {s_: sum((a_, k_, s_) in _done for a_ in ARMS for k_ in range(len(DAYS))) for s_ in SEEDS}
print("SEED-CHUNK STATUS (finished runs / " + str(len(ARMS) * len(DAYS)) + " per seed): " + " | ".join(
    f"seed {s_}: {n_}" for s_, n_ in _per_seed.items()), flush=True)
_next = [s_ for s_, n_ in _per_seed.items() if n_ < len(ARMS) * len(DAYS)]
print("NEXT CHUNK: " + (f'RUN_SUBSET = {{"seeds": [{_next[0]}]}}' if _next else
      'all seeds finished -> final chunk RUN_SUBSET = {"seeds": []} (analysis only)'), flush=True)
_todo = [r for r in PLAN if r not in _done]
_scale = lambda a: (R2_EPOCHS_TO_RUN / R2_EPOCHS) if a == "R2" else (EPOCHS_TO_RUN / EPOCHS)
_est = sum(SEC_PER_RUN[a] * _scale(a) for a, _, _ in _todo)
print(f"{len(PLAN)} runs planned ({len(ARMS_TO_RUN)} arms x {len(FOLDS_TO_RUN)} folds x {len(SEEDS_TO_RUN)} seeds) | "
      f"already done: {len(PLAN) - len(_todo)} | to train now: {len(_todo)}")
for _a in ARMS_TO_RUN:                                # E23: per-arm estimate
    _na = sum(1 for a_, _, _ in _todo if a_ == _a)
    print(f"  estimate {_a:<5}: {_na:>3} runs x {SEC_PER_RUN[_a] * _scale(_a):>4.0f} s = "
          f"{_na * SEC_PER_RUN[_a] * _scale(_a) / 60:5.1f} min")
print(f"ESTIMATED training time: {_est / 3600:.2f} h (+ parsing on a first session) — budget {BUDGET_HOURS} h")
if _est / 3600 > BUDGET_HOURS:
    loud(f"The estimate {_est / 3600:.2f} h exceeds the {BUDGET_HOURS} h budget. Nothing is reduced automatically:\n"
         "set RUN_SUBSET (top cell) to run part of the plan now; finished runs are skipped when you resume.")
    _per_seed_h = sum(SEC_PER_RUN[a_] * _scale(a_) for a_ in ARMS_TO_RUN) * len(FOLDS_TO_RUN) / 3600
    print("SEED-CHUNK PLAN (one Save Version each; attach the previous chunk's output as RESTORE_FROM): " +
          " -> ".join(f'{{"seeds": [{s_}]}} (~{_per_seed_h:.1f} h)' for s_ in SEEDS) +
          ' -> {"seeds": []} (analysis only)', flush=True)
print("Note: mamba-ssm's CUDA kernels are not bit-deterministic, so repeated B0 runs can differ slightly; every arm of a "
      "(fold, seed) uses the SAME B0 checkpoint, so the residual comparisons are unaffected by this noise.")
_t0, _n = time.time(), 0
for (arm, k, s) in PLAN:
    if (arm, k, s) in _done:
        print(f"[{arm} fold {k} seed {s}] already in {RESULTS_CSV.name} — skipped")
        continue
    for dep in DEPENDS.get(arm, []):
        assert (dep, k, s) in _done or (dep, k, s) in load_done(), f"{arm} fold {k} seed {s}: dependency {dep} missing"
    res = run_arm_e23(arm, k, s, epochs=EPOCHS_TO_RUN, r2_epochs=R2_EPOCHS_TO_RUN)
    append_row(res)
    _done[(arm, k, s)] = res
    _n += 1
    _el = time.time() - _t0
    print(f"[{_n}/{len(_todo)}] {arm} fold {k} ({res['fold_day']}) seed {s} | window F1 {res['window_F1']:.4f} | mission F1 "
          f"{res['mission_F1']:.4f} | epoch {res['selected_epoch']}" + (f" | w {res['b1_w']}" if arm in ("B1", "B1c") else "")
          + f" | {res['train_seconds']:.0f} s | ETA {(len(_todo) - _n) * _el / _n / 60:.0f} min")
    mem_report(f"after {arm} fold {k} seed {s}")
print(f"\nTraining finished in {(time.time() - _t0) / 60:.1f} min | runs in CSV: {len(load_done())}/{len(PLAN)}")
mem_report("end of training loop")
if CLEAN_DISK_AFTER_TRAINING and not SMOKE and len(load_done()) >= len(PLAN):
    _freed = 0
    for _d in [CACHE_DIR] + ([BASE] if str(DATA_ROOT).startswith(str(BASE)) else []):
        if Path(_d).exists() and str(Path(_d)).startswith(str(OUT)):      # only ever inside /kaggle/working
            _freed += sum(f.stat().st_size for f in Path(_d).rglob("*") if f.is_file())
            shutil.rmtree(_d, ignore_errors=True)
    print(f"CLEAN_DISK_AFTER_TRAINING: removed the parse cache and the raw dataset clone ({_freed / 1e9:.1f} GB); the "
          "analysis uses the in-memory arrays only", flush=True)
    mem_report("after disk clean-up")

# ---- run-level hooks (asserted per run; summarised here) ----
_df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=CSV_FIELDS)
_same = _df.groupby(["fold", "seed"]).id_hash.nunique().max() if len(_df) else 1
hook(5, _same == 1, f"test window ids + test missions + training pool identical across arms in every (fold, seed): max "
                    f"distinct hashes {_same}", stop=True)
_res_rows = _df[_df.arm.isin(["R1", "C1", "C2", "D1", "D1r", "D2", "D1sh", "D2sh"])]
hook(7, bool(len(_res_rows) == 0 or (_res_rows.zero_init_maxdiff < ZERO_INIT_TOL).all()),
     f"{len(_res_rows)} residual runs: max |z_final - z_IMU| at step 0 = {_res_rows.zero_init_maxdiff.max() if len(_res_rows) else 'n/a'}",
     stop=True)
hook(8, bool(len(_res_rows) == 0 or ((_res_rows.anchor_hash_before == _res_rows.anchor_hash_after).all()
                                     and (_res_rows.anchor_repro_maxdiff < ANCHOR_REPRO_TOL).all())),
     f"frozen anchor unchanged in all {len(_res_rows)} R1/C1/C2 runs; anchor reproduces B0 test probabilities (max |diff| "
     f"{_res_rows.anchor_repro_maxdiff.max() if len(_res_rows) else 'n/a'})", stop=True)
hook(9, True, f"alpha within [0, {ALPHA_MAX}] asserted on every evaluation pass ({sum(ALPHA_CHECKS):,} windows checked this "
              "session)", stop=True)
_c2 = _df[_df.arm.isin(["C2", "D1sh", "D2sh"])]
hook(10, bool(len(_c2) == 0 or (_c2.c2_nonidentity_min > 0.95).all()) and all(C1_CHECKS),
     f"C1: no audio encoder and output unchanged with zeroed audio ({len(C1_CHECKS)} runs this session); "
     f"C2: permutation stays inside train / val / test (asserted), min non-identity share over splits and runs "
     f"{_c2.c2_nonidentity_min.min() if len(_c2) else 'n/a'}; labels hash unchanged ({Y_HASH})", stop=True)

## 9. Results (all arms on the identical full window set; seeds 11–51)
- **Primary** (K = 2): D1 and D2, each vs R1, under 7 conditions. Condition 7 is "beats its own shuffled control".
- **Secondary** (conditions 1–5): D2 vs D1, D1 vs D1r, and each of R1, D1, D2, D1sh, D2sh vs C1.
- **Also:** eval-time path ablation for D1 and D2, per-day F1, per-class recall and residual diagnostics.

In [ ]:
# ==============================================================================
# 6a. RESULT HELPERS — audio reference functions (verbatim) + E18 pooling, rule, diagnostics
# ==============================================================================
def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Resample MISSIONS within class (never windows). Returns bootstrap confusion matrices (B, 5, 5)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    cnt = np.zeros((len(ms), N_CLASSES), int)
    pos = {m: i for i, m in enumerate(ms)}
    mcls = np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        draw = rows[rng.integers(0, len(rows), (B, len(rows)))]
        cms[:, c, :] = cnt[draw].sum(1)
    return cms


def boot_summary(missions, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    p, r, f = prf_from_cm(cm)
    cms = cluster_bootstrap(missions, y, pred)
    bp, br, bf = zip(*(prf_from_cm(c) for c in cms))
    bp, br, bf = np.array(bp), np.array(br), np.array(bf)
    ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
    tab = pd.DataFrame({"precision": p, "P CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bp).T],
                        "recall": r, "R CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(br).T],
                        "f1": f, "F1 CI": [f"[{a:.3f}, {b:.3f}]" for a, b in ci(bf).T]},
                       index=[f"class {c}" for c in range(N_CLASSES)])
    macro = (f.mean(), *ci(bf.mean(1)))
    bal = (r.mean(), *ci(br.mean(1)))
    return tab, macro, bal, cm, (r, ci(br))


def macro_f1(yt, pred):
    return float(f1_score(yt, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0))


def pool_arm(arm):
    per_seed, ids0, meta = {}, None, None
    for s in SEEDS:
        parts = [dict(np.load(PROBS_DIR / f"{arm}_{k}_{s}.npz")) for k in range(len(DAYS))]
        ids = np.concatenate([p["window_id"] for p in parts])
        assert len(ids) == len(set(ids)) == len(WID), f"{arm} seed {s}: pooled windows"
        order = np.argsort(ids)
        per_seed[s] = np.concatenate([p["probs"] for p in parts])[order]
        if ids0 is None:
            ids0 = ids[order]
            meta = pd.DataFrame({c: np.concatenate([p[c] for p in parts])[order] for c in ("mission", "cls", "day")})
        assert np.array_equal(ids[order], ids0)
    return ids0, meta, per_seed


def arm_stats(meta, per_seed):
    yt = meta.cls.values.astype(int)
    P = np.mean(list(per_seed.values()), axis=0)
    pred = P.argmax(1)
    _, macro, bal, cm, (rec, _) = boot_summary(meta.mission.values, yt, pred)
    my_, mP_, mn_ = mission_level(meta.mission.values, yt, P)
    _, mmacro, _, _, _ = boot_summary(mn_, my_, mP_.argmax(1))
    fold_f1 = {d: macro_f1(yt[(meta.day == d).values], pred[(meta.day == d).values]) for d in DAYS}
    return {"window_f1": macro, "mission_f1": mmacro, "recall": rec, "fold_f1": fold_f1, "min_fold": min(fold_f1.values()),
            "worst_day": min(fold_f1, key=fold_f1.get), "seed_f1": {s: macro_f1(yt, v.argmax(1)) for s, v in per_seed.items()},
            "c4_recall": float(rec[4]), "fcn": float((pred[yt > 0] == 0).mean()), "pred": pred, "y": yt,
            "boot": np.array([prf_from_cm(c)[2].mean() for c in cluster_bootstrap(meta.mission.values, yt, pred)])}


def compare(st, ref):
    """Paired comparison of an arm with a reference arm under the pre-registered conditions."""
    delta = st["window_f1"][0] - ref["window_f1"][0]
    q = (100 - PAIRED_CI) / 2
    lo, hi = np.percentile(st["boot"] - ref["boot"], [q, 100 - q])
    fd = {d: st["fold_f1"][d] - ref["fold_f1"][d] for d in DAYS}
    sd_ = {s: st["seed_f1"][s] - ref["seed_f1"][s] for s in ref["seed_f1"]}
    c = {"c1_delta": delta >= MIN_DELTA, "c2_ci": lo > 0, "c3_folds": sum(v > 0 for v in fd.values()) >= MIN_FOLDS_SAME_SIGN,
         "c4_seeds": sum(v > 0 for v in sd_.values()) >= MIN_SEEDS_POSITIVE, "c5_worst_day": st["min_fold"] >= ref["min_fold"],
         "c6_no_harm": (st["c4_recall"] >= ref["c4_recall"] - NO_HARM_C4_RECALL) and (st["fcn"] <= ref["fcn"] + NO_HARM_FCN)}
    worse = delta <= -MIN_DELTA and hi < 0 and sum(v < 0 for v in fd.values()) >= MIN_FOLDS_SAME_SIGN
    return {"delta": float(delta), "ci_lo": float(lo), "ci_hi": float(hi), "folds_pos": sum(v > 0 for v in fd.values()),
            "seeds_pos": sum(v > 0 for v in sd_.values()), "min_fold": st["min_fold"], "min_fold_ref": ref["min_fold"],
            "c4_recall": st["c4_recall"], "c4_recall_ref": ref["c4_recall"], "fcn": st["fcn"], "fcn_ref": ref["fcn"],
            "mission_delta": float(st["mission_f1"][0] - ref["mission_f1"][0]), **c, "worse": worse,
            "fold_delta": fd, "seed_delta": sd_}


def verdict_vs_b0(c):
    if all(c[k] for k in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds", "c5_worst_day", "c6_no_harm")):
        return "CREDITED"
    return "WORSE" if c["worse"] else "NOT CREDITED"


def verdict_vs_b1(c):
    return "BEATS WEIGHTED FUSION" if all(c[k] for k in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds", "c5_worst_day")) \
        else "DOES NOT BEAT WEIGHTED FUSION"


def residual_diagnostics(arm, k, s):
    z = np.load(PROBS_DIR / f"{arm}_{k}_{s}.npz")
    zb = np.load(PROBS_DIR / f"B0_{k}_{s}.npz")
    assert np.array_equal(z["window_id"], zb["window_id"])
    yt, pf, pa = z["cls"].astype(int), z["probs"].argmax(1), zb["probs"].argmax(1)
    corr, brok = (pa != yt) & (pf == yt), (pa == yt) & (pf != yt)
    adj = np.abs(pa - yt) == 1
    row = {"arm": arm, "fold": k, "fold_day": DAYS[k], "seed": s, "mean_alpha": float(z["alpha"].mean()),
           "mean_alpha_dz_norm": float(z["corr_norm"].mean()), "share_argmax_differs": float((pf != pa).mean()),
           "corrected": int(corr.sum()), "broken": int(brok.sum()), "net": int(corr.sum() - brok.sum()),
           "corrected_anchor_adjacent_err": int((corr & adj).sum()), "corrected_anchor_distant_err": int((corr & ~adj).sum()),
           "broken_to_adjacent_err": int((brok & (np.abs(pf - yt) == 1)).sum()),
           "broken_to_distant_err": int((brok & (np.abs(pf - yt) >= 2)).sum())}
    for c in range(N_CLASSES):
        row[f"corrected_c{c}"] = int((corr & (yt == c)).sum())
        row[f"broken_c{c}"] = int((brok & (yt == c)).sum())
    return row

In [ ]:
# ==============================================================================
# 9b. E23 ANALYSIS: pooled results; primary (7 conditions) / secondary contrasts in code; path ablation; per-day;
#     residual diagnostics; parameters; SOFT HOOK 12
# ==============================================================================
mem_report("start of 9b analysis")
_done = load_done()
COMPLETE = [a for a in ARMS if all((a, k, s) in _done for k in range(len(DAYS)) for s in SEEDS)]
STATS, CON, VERDICT, ABL = {}, {}, {}, {}
TABLE1 = TABLE2 = PERFOLD = DIAG = ABL_TABLE = None
PRIMARY = [("D1 vs R1", "D1", "R1", "D1sh"), ("D2 vs R1", "D2", "R1", "D2sh")]
ATTRIBUTION = [("D1 vs D1sh", "D1", "D1sh"), ("D2 vs D2sh", "D2", "D2sh")]
SECONDARY = [("D2 vs D1", "D2", "D1"), ("D1 vs D1r", "D1", "D1r"), ("R1 vs C1", "R1", "C1"), ("D1 vs C1", "D1", "C1"),
             ("D2 vs C1", "D2", "C1"), ("D1sh vs C1", "D1sh", "C1"), ("D2sh vs C1", "D2sh", "C1")]


def conds(c_, min_delta):
    """The rule's conditions for one paired comparison at threshold `min_delta` (E18b's compare() supplies the rest)."""
    return {"c1_delta": c_["delta"] >= min_delta, "c2_ci": c_["ci_lo"] > 0, "c3_folds": c_["folds_pos"] >= MIN_FOLDS_SAME_SIGN,
            "c4_seeds": c_["seeds_pos"] >= MIN_SEEDS_POSITIVE, "c5_worst_day": bool(c_["c5_worst_day"]),
            "c6_no_harm": bool(c_["c6_no_harm"]),
            "worse": (c_["delta"] <= -min_delta and c_["ci_hi"] < 0
                      and sum(v < 0 for v in c_["fold_delta"].values()) >= MIN_FOLDS_SAME_SIGN)}


def pooled_key(arm, key):
    """Seed-averaged pooled probabilities stored under `key` (e.g. the path-ablation outputs)."""
    per_seed, ids0, y0 = [], None, None
    for s in SEEDS:
        parts = [dict(np.load(PROBS_DIR / f"{arm}_{k}_{s}.npz")) for k in range(len(DAYS))]
        ids = np.concatenate([p["window_id"] for p in parts])
        order = np.argsort(ids)
        per_seed.append(np.concatenate([p[key] for p in parts])[order])
        if ids0 is None:
            ids0, y0 = ids[order], np.concatenate([p["cls"] for p in parts])[order]
    return np.mean(per_seed, axis=0), y0


if SMOKE:
    print("SMOKE run: pooled analysis skipped (it needs 5 folds x 5 seeds). Smoke rows:")
    print(_df[["arm", "fold", "seed", "window_F1", "mission_F1", "selected_epoch", "mean_alpha", "zero_init_maxdiff",
               "anchor_repro_maxdiff", "abl_strong_window_F1", "abl_weak_window_F1", "train_seconds"]].to_string(index=False))
elif not COMPLETE:
    loud("No arm is complete for all 5 folds x 5 seeds yet. This is expected after a seed chunk: attach this version's "
         "output as RESTORE_FROM and run the next chunk (see NEXT CHUNK in the training cell); the last chunk is "
         "RUN_SUBSET = {'seeds': []}.")
else:
    _incomplete = [a for a in ARMS if a not in COMPLETE]
    if _incomplete:
        loud(f"Incomplete arms (not analysed): {_incomplete}")
    ids_ref = None
    for arm in COMPLETE:
        ids, meta, per_seed = pool_arm(arm)
        ids_ref = ids if ids_ref is None else ids_ref
        assert np.array_equal(ids, ids_ref), f"{arm}: different pooled windows — not paired"
        STATS[arm] = arm_stats(meta, per_seed)
    for name, a, b, *_ in PRIMARY + ATTRIBUTION + SECONDARY:
        if a in STATS and b in STATS:
            CON[name] = compare(STATS[a], STATS[b])
    # ---- primary verdicts (conditions 1-6 at +0.01, condition 7 = vs own shuffled control at +0.02) ----
    for name, a, b, sh in PRIMARY:
        att = f"{a} vs {sh}"
        if name not in CON or att not in CON:
            VERDICT[name] = "INCOMPLETE"
            continue
        c16, c7 = conds(CON[name], MIN_DELTA_PRIMARY), conds(CON[att], ATTR_MIN_DELTA)
        CON[name]["c7_attribution"] = all(c7[k_] for k_ in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds"))
        CON[name]["conds"] = c16
        VERDICT[name] = ("CREDITED" if all(c16[k_] for k_ in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds", "c5_worst_day",
                                                                 "c6_no_harm")) and CON[name]["c7_attribution"]
                         else "WORSE" if c16["worse"] else "NOT CREDITED")
    for name, a, b in ATTRIBUTION + SECONDARY:
        if name in CON:
            c15 = conds(CON[name], MIN_DELTA_PRIMARY)
            CON[name]["conds"] = c15
            VERDICT[name] = ("MEETS CONDITIONS 1-5" if all(c15[k_] for k_ in ("c1_delta", "c2_ci", "c3_folds", "c4_seeds",
                                                                                "c5_worst_day"))
                             else "WORSE (mirror)" if c15["worse"] else "DOES NOT MEET CONDITIONS 1-5")
        else:
            VERDICT[name] = "INCOMPLETE"
    # ---- tables ----
    _t1 = []
    for arm in COMPLETE:
        st = STATS[arm]
        _t1.append({"arm": arm, "name": ARM_NAMES[arm], "trainable_params": int(PARAMS.loc[arm, "trainable_params"]),
                    "window F1 [95% CI]": f"{st['window_f1'][0]:.4f} [{st['window_f1'][1]:.4f}, {st['window_f1'][2]:.4f}]",
                    "mission F1 [95% CI]": f"{st['mission_f1'][0]:.4f} [{st['mission_f1'][1]:.4f}, {st['mission_f1'][2]:.4f}]",
                    **{f"seed {s} F1": round(v, 4) for s, v in st["seed_f1"].items()},
                    "worst day": f"{st['worst_day']} ({st['min_fold']:.4f})",
                    **{f"F1 {d}": round(v, 4) for d, v in st["fold_f1"].items()},
                    **{f"recall c{c_}": round(float(st["recall"][c_]), 4) for c_ in range(N_CLASSES)},
                    "faulty called Normal": round(st["fcn"], 4)})
    TABLE1 = pd.DataFrame(_t1)
    _t2 = []
    for kind, group in (("PRIMARY", [(n, a, b) for n, a, b, _ in PRIMARY]), ("ATTRIBUTION (condition 7)", ATTRIBUTION),
                        ("SECONDARY", SECONDARY)):
        for name, a, b in group:
            if name not in CON:
                _t2.append({"contrast": name, "kind": kind, "verdict": "INCOMPLETE"})
                continue
            c_ = CON[name]
            _t2.append({"contrast": name, "kind": kind, "delta": round(c_["delta"], 4), "CI_lo_99": round(c_["ci_lo"], 4),
                        "CI_hi_99": round(c_["ci_hi"], 4), "folds_positive": c_["folds_pos"], "seeds_positive": c_["seeds_pos"],
                        "worst_day_F1": round(c_["min_fold"], 4), "worst_day_F1_ref": round(c_["min_fold_ref"], 4),
                        "class4_recall": round(c_["c4_recall"], 4), "class4_recall_ref": round(c_["c4_recall_ref"], 4),
                        "faulty_called_normal": round(c_["fcn"], 4), "faulty_called_normal_ref": round(c_["fcn_ref"], 4),
                        "delta_mission_F1": round(c_["mission_delta"], 4),
                        **{k_: bool(v_) for k_, v_ in c_["conds"].items()},
                        "c7_attribution": c_.get("c7_attribution", ""),
                        "meets_+0.02_line": c_["delta"] >= MIN_DELTA_STRICT, "verdict": VERDICT[name]})
    TABLE2 = pd.DataFrame(_t2)
    PERFOLD = pd.DataFrame([{"arm": a, "fold_day": d, "window_F1_seedavg": STATS[a]["fold_f1"][d]}
                            for a in COMPLETE for d in DAYS])
    _ab = []
    for arm in [a for a in ("D1", "D2", "D1r", "D1sh", "D2sh") if a in COMPLETE]:
        full = STATS[arm]["window_f1"][0]
        row = {"arm": arm, "window_F1_full": full}
        for p in PATHS:
            Pp, yy = pooled_key(arm, f"probs_abl_{p}")
            row[f"window_F1_{p}_path_zeroed"] = macro_f1(yy, Pp.argmax(1))
            row[f"drop_when_{p}_zeroed"] = full - row[f"window_F1_{p}_path_zeroed"]
        ds, dw = row["drop_when_strong_zeroed"], row["drop_when_weak_zeroed"]
        row["reading"] = ("neither path alone is needed (drops < 0.005)" if max(ds, dw) < 0.005 else
                          "the STRONG path carries the signal" if ds > 2 * max(dw, 1e-9) else
                          "the WEAK path carries the signal" if dw > 2 * max(ds, 1e-9) else "both paths contribute")
        ABL[arm] = row
        _ab.append(row)
    ABL_TABLE = pd.DataFrame(_ab)
    DIAG = pd.DataFrame([residual_diagnostics(a, k, s) for a in ("C1", "R1", "D1", "D1r", "D2", "D1sh", "D2sh")
                         if a in COMPLETE for k in range(len(DAYS)) for s in SEEDS])
    TABLE1.to_csv(OUT / "e23_table_results.csv", index=False)
    TABLE2.to_csv(OUT / "e23_table_paired.csv", index=False)
    PERFOLD.to_csv(OUT / "e23_perfold.csv", index=False)
    ABL_TABLE.to_csv(OUT / "e23_path_ablation.csv", index=False)
    DIAG.to_csv(OUT / "e23_residual_diagnostics.csv", index=False)
    pd.set_option("display.width", 250)
    print("=== Pooled out-of-fold results (seed-averaged, full v29 window set) ===")
    print(TABLE1.set_index("arm").T.to_string())
    print(f"\n=== Contrasts ({PAIRED_CI:.0f}% paired CIs) and verdicts from the pre-registered rule ===")
    print(TABLE2.set_index(["kind", "contrast"])[[c_ for c_ in ("delta", "CI_lo_99", "CI_hi_99", "folds_positive",
                                                               "seeds_positive", "c5_worst_day", "c6_no_harm",
                                                               "c7_attribution", "meets_+0.02_line", "verdict")
                                                  if c_ in TABLE2.columns]].to_string())
    print("\n=== Per-day window F1 (seed-averaged) ===")
    print(PERFOLD.pivot(index="arm", columns="fold_day", values="window_F1_seedavg").reindex(COMPLETE).round(4).to_string())
    if len(ABL_TABLE):
        print("\n=== Eval-time path ablation (diagnostic) ===")
        print(ABL_TABLE.round(4).to_string(index=False))
    if len(DIAG):
        print("\n=== Residual diagnostics (sums over 5 folds x 5 seeds; mean alpha averaged) ===")
        print(DIAG.groupby("arm")[["corrected", "broken", "net"]].sum()
              .join(DIAG.groupby("arm")[["mean_alpha"]].mean()).to_string())

    # ---- hook 12 (SOFT): B0 and R1 per-fold means vs E18b (bounded search for an attached e18b_results.csv) ----
    _cands = [OUT / "e18b_results.csv"] + [Path(p) for depth in range(1, 4)
                                           for p in sorted(glob.glob("/kaggle/input/" + "/".join(["*"] * depth) + "/e18b_results.csv"))]
    _cands = [p for p in _cands if p.exists()]
    if _cands:
        _e = pd.read_csv(_cands[0])
        _msgs, _ok12 = [], True
        for a in ("B0", "R1"):
            if a in COMPLETE:
                ref_ = _e[_e.arm == a].groupby("fold").window_F1.mean().reindex(range(len(DAYS))).values
                cur_ = _df[_df.arm == a].groupby("fold").window_F1.mean().reindex(range(len(DAYS))).values
                dev_ = cur_ - ref_
                _ok12 &= bool(np.all(np.abs(dev_) <= E18B_REF_TOL))
                _msgs.append(f"{a}: " + ", ".join(f"{DAYS[i]} {dev_[i]:+.4f}" for i in range(len(DAYS))))
        hook(12, _ok12, f"vs E18b ({_cands[0]}): " + " | ".join(_msgs)
             + ("" if _ok12 else f" — WARN: outside ±{E18B_REF_TOL} (mamba-ssm CUDA non-determinism)"), stop=False)
    else:
        print("Soft hook 12 not run: no e18b_results.csv attached.")

In [ ]:
# ==============================================================================
# 10. SUMMARY (generated from the numbers; verdicts come from the rule in code) -> e23_summary.txt; zip; hooks
# ==============================================================================
import zipfile

mem_report("start of 10 summary / zip")
_L = ["WHAT E23 TELLS US (dual-path cross-attention + wavelet scattering; full v29 windows; seeds " + str(SEEDS) + ")", "",
      f"Scattering backend: {SCAT_BACKEND}; {N_SCAT_PATHS} paths -> token size {N_CH * N_SCAT_PATHS}. Scattering "
      "frequencies are normalised (cycles/sample), not Hz: the audio sample rate is unverified.",
      "Trainable parameters per arm: " + ", ".join(f"{a} {int(PARAMS.loc[a, 'trainable_params']):,}" for a in ARMS) + ".", ""]
if SMOKE:
    _L += ["SMOKE run only (one fold, one seed, 2 epochs, every arm): checks that the pipeline runs end to end. No "
           "scientific conclusion can be drawn from it."]
elif not STATS:
    _L += ["No arm complete yet: no model results. Run the next seed chunk (the notebook resumes)."]
else:
    for arm in COMPLETE:
        st = STATS[arm]
        _L.append(f"{arm} ({ARM_NAMES[arm]}): window F1 {st['window_f1'][0]:.4f} [{st['window_f1'][1]:.4f}, "
                  f"{st['window_f1'][2]:.4f}], mission F1 {st['mission_f1'][0]:.4f}, worst day {st['worst_day']} "
                  f"({st['min_fold']:.4f}), Class-4 recall {st['c4_recall']:.3f}, faulty called Normal {st['fcn']:.4f}")
    _L += ["", "PRIMARY (K = 2; conditions 1-6 at +0.01 vs R1, condition 7 = beats its own shuffled control by +0.02):"]
    for name, a, b, sh in PRIMARY:
        c_ = CON.get(name)
        _L.append(f"- {name}: " + (f"delta {c_['delta']:+.4f} [{c_['ci_lo']:+.4f}, {c_['ci_hi']:+.4f}] "
                                   f"({PAIRED_CI:.0f}% paired CI), folds+ {c_['folds_pos']}/5, seeds+ {c_['seeds_pos']}/5, "
                                   f"attribution vs {sh}: {c_.get('c7_attribution')}, +0.02 line: "
                                   f"{'met' if c_['delta'] >= MIN_DELTA_STRICT else 'not met'} -> " if c_ else "")
                  + VERDICT.get(name, "INCOMPLETE"))
    _L += ["", "ATTRIBUTION and SECONDARY (conditions 1-5, no multiplicity claim):"]
    for name, a, b in ATTRIBUTION + SECONDARY:
        c_ = CON.get(name)
        _L.append(f"- {name}: " + (f"delta {c_['delta']:+.4f} [{c_['ci_lo']:+.4f}, {c_['ci_hi']:+.4f}], folds+ "
                                   f"{c_['folds_pos']}/5, seeds+ {c_['seeds_pos']}/5 "
                                   f"({'>=' if c_['delta'] >= MIN_DELTA_STRICT else '<'} +0.02) -> " if c_ else "")
                  + VERDICT.get(name, "INCOMPLETE"))
    _cred = [n for n, *_ in PRIMARY if VERDICT.get(n) == "CREDITED"]
    _L.append("-> " + (f"CREDITED: {', '.join(_cred)}." if _cred else
                       "No dual-path candidate is credited by the pre-registered rule."))
    if ABL:
        _L += ["", "Path ablation (diagnostic): " + "; ".join(
            f"{a}: zeroing strong {r['drop_when_strong_zeroed']:+.4f}, zeroing weak {r['drop_when_weak_zeroed']:+.4f} "
            f"-> {r['reading']}" for a, r in ABL.items())]
    if DIAG is not None and len(DIAG):
        _g = DIAG.groupby("arm")
        _L.append("Residual corrections (sum over folds x seeds): " + "; ".join(
            f"{a}: corrected {int(g.corrected.sum())}, broken {int(g.broken.sum())}, net {int(g.net.sum()):+d}, mean alpha "
            f"{g.mean_alpha.mean():.3f}" for a, g in _g))
_L += ["",
       "Limits",
       "- Same 95 missions and 5 recording days as E18b (one day, 2021-12-22, has only 10 missions): the seeds test "
       "training randomness, not new data.",
       "- Scattering frequencies are normalised (cycles/sample), not Hz; the audio sample rate is unverified.",
       "- The D arms differ in capacity from R1 (parameter counts above); they are therefore also judged against their "
       "own shuffled-audio controls (condition 7).",
       "- Only one attention layer per path and one strong/weak split rule (top half by fit-window energy) were tried; "
       "nothing was tuned on any fold.",
       "- mamba-ssm CUDA kernels are not bit-deterministic; B0/R1 vs E18b is only a soft check.",
       "- The +0.01 threshold for condition 1 was chosen before the run because the audio-specific ceiling of R1 is "
       "about +0.02; each contrast is also reported against +0.02."]
SUMMARY = "\n".join(_L)
print(SUMMARY)
(OUT / ("e23_summary_smoke.txt" if SMOKE else "e23_summary.txt")).write_text(SUMMARY, encoding="utf-8")

_zip = OUT / ("e23_outputs_smoke.zip" if SMOKE else "e23_outputs.zip")
_n = 0
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("e23_*.csv")) + sorted(OUT.glob("e23_*.txt")) + sorted(PROBS_DIR.glob("*.npz")):
        zf.write(p, p.relative_to(OUT))
        _n += 1
print(f"\nWrote {_zip} ({_n} files, {_zip.stat().st_size / 1e6:.1f} MB; caches, checkpoints and the dataset excluded)")

print("\n" + "=" * 110)
print("VERIFICATION HOOKS (hook 12 is SOFT: WARN only)")
print("=" * 110)
for n in sorted(HOOK_NAMES):
    res = HOOKS.get(n, [])
    status = "NOT RUN" if not res else ("PASS" if all(ok for ok, _ in res) else ("WARN" if n == 12 else "FAIL"))
    print(f"  HOOK {n:<2} {status:<8} {HOOK_NAMES[n]}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else ('WARN' if n == 12 else 'FAIL')}: {msg}")
print("=" * 110)
print("\n💾 Click \"Save Version\" -> \"Save & Run All (Commit)\" to keep /kaggle/working (or download e23_outputs.zip).")